# Evidence-weighted winning ensemble for Parkinson's FoG

This notebook targets the competition's exact mean average precision
over `StartHesitation`, `Turn`, and `Walking`.  It is a new workflow;
it does not overwrite the dissertation experiments.

The design follows the strongest reproducible evidence rather than
indiscriminately stacking every idea:

1. **DeFOG event-assisted specialist.** The competition supplies the
   FoG interval boundaries in `events.csv` and activity intervals in
   `tasks.csv`, including hidden DeFOG test records.  We classify each
   known event with task, duration/position, sensor summaries, clinical
   metadata, and optional pooled neural scores, then broadcast the
   probabilities only over that event.  Cross-validation retains the
   supplied intervals but hides `Type`, exactly simulating test use.
2. **First-place family:** separate long-context patch Transformer →
   BiLSTM models per domain, per-series normalization, an auxiliary
   `Event` output for DeFOG, and dense overlap averaging.
3. **Second-place family:** a complementary two-layer BiGRU using raw,
   lag, lead, and (for TDCS) cumulative-sum features, with longer
   inference than training and center stitching.
4. **Local evidence:** natural sampling plus square-root weighted BCE is
   the DeFOG default; a positive-aware, full-inverse TDCS `Turn`
   specialist is blended only for that target.
5. **Leakage control:** immutable subject folds, native-row OOF scoring,
   predeclared evidence-weighted blending, train/test overlap exclusion, and
   fold-scoped artifacts.

`smoke` proves the mechanics on small models.  `competition` is the
recommended Kaggle-GPU preset.  `high` and `near_exact` are expensive
reproductions, not promises of a better score.  The historical
competition is closed, so this can target a late/historical leaderboard
result but cannot win the original prize.

## 1. Setup, profiles, and run contract

The notebook never calls the internet and discovers either the local
repository layout or Kaggle's mounted competition directory.  For a
serious run, use a GPU, enable persistence, and finish all folds before
trusting OOF or submission output.

In [ ]:
from __future__ import annotations

print("[FoG] Notebook started — if you can see this, the Kaggle session is connected.", flush=True)

from collections import OrderedDict
from dataclasses import asdict, dataclass, field, replace
from itertools import product
from pathlib import Path
import gc
import hashlib
import json
import math
import marshal
import os
import random
import sys
import time
import warnings

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")

print("[FoG] Importing NumPy, pandas, scikit-learn and TensorFlow…", flush=True)

import numpy as np
import pandas as pd
import scipy
from scipy.signal import periodogram, welch
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sklearn
import tensorflow as tf
from IPython.display import display

print(f"[FoG] Imports complete. TensorFlow {tf.__version__}.", flush=True)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)

TARGETS = ["StartHesitation", "Turn", "Walking"]
AXES = ["AccV", "AccML", "AccAP"]
TARGET_TO_INDEX = {name: i for i, name in enumerate(TARGETS)}
DOMAIN = {
    "defog": {"hz": 100, "metadata": "defog_metadata.csv"},
    "tdcsfog": {"hz": 128, "metadata": "tdcsfog_metadata.csv"},
}
EPS = np.float32(1e-6)
SEED = 42

@dataclass(frozen=True)
class Profile:
    name: str
    n_folds: int
    folds: tuple[int, ...]
    seeds: tuple[int, ...]
    max_records_per_domain: int | None
    epochs_patch: int
    epochs_bigru: int
    steps_per_epoch: int
    batch_patch: int
    batch_bigru: int
    patch_width: int
    patch_heads: int
    patch_layers: int
    recurrent_units: int
    patch_tokens: int
    inference_overlap: float
    active_branches: tuple[str, ...]
    augment: bool = False

PROFILES = {
    "smoke": Profile(
        "smoke", 2, (0, 1), (42,), 12, 1, 1, 24, 2, 4,
        48, 4, 1, 48, 64, 0.50,
        ("patch_bilstm", "engineered_bigru", "tdcs_turn_specialist"),
        False,
    ),
    "competition": Profile(
        "competition", 5, (0, 1, 2, 3, 4), (42,), None,
        12, 8, 120, 4, 12, 192, 6, 3, 192, 864, 0.75,
        ("patch_bilstm", "engineered_bigru", "tdcs_turn_specialist"),
        False,
    ),
    "high": Profile(
        "high", 5, (0, 1, 2, 3, 4), (42, 31415), None,
        24, 12, 240, 3, 8, 256, 8, 4, 256, 864, 0.875,
        ("patch_bilstm", "engineered_bigru", "tdcs_turn_specialist"),
        True,
    ),
    # Near-exact uses stock Keras attention, so width=320/heads=8;
    # the winner's custom 320/6 projection is not falsely claimed.
    "near_exact": Profile(
        "near_exact", 5, (0, 1, 2, 3, 4), (42, 31415), None,
        60, 18, 400, 2, 6, 320, 8, 5, 320, 864, 0.9375,
        ("patch_bilstm", "engineered_bigru", "tdcs_turn_specialist"),
        True,
    ),
}

# Visible Kaggle default: a Save & Run All creates the real model and
# submission. Set FOG_PROFILE=smoke explicitly for pipeline tests.
RUN_PROFILE = os.environ.get("FOG_PROFILE", "competition")
# KAGGLE RUN CONTROL: for a serious GPU run, edit the default above to
# "competition" (or set FOG_PROFILE=competition) and Run All.
if RUN_PROFILE not in PROFILES:
    raise ValueError(f"Unknown FOG_PROFILE={RUN_PROFILE!r}")
CFG = PROFILES[RUN_PROFILE]
RUN_NEURAL = os.environ.get("FOG_RUN_NEURAL", "1") == "1"
RUN_FINAL_SUBMISSION = os.environ.get(
    "FOG_RUN_SUBMISSION", "1" if RUN_PROFILE != "smoke" else "0"
) == "1"
REUSE_COMPLETED = os.environ.get("FOG_REUSE_COMPLETED", "1") == "1"
RUN_EXPLORATORY_META_BLEND = os.environ.get("FOG_EXPLORATORY_META_BLEND", "0") == "1"
MAX_WALL_HOURS = float(os.environ.get(
    "FOG_MAX_WALL_HOURS", "8.2" if RUN_PROFILE == "competition" else "72"
))
RUN_STARTED = time.time()

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

seed_everything(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")

def locate_layout():
    kaggle_input = Path("/kaggle/input")
    mounted = (sorted((p for p in kaggle_input.iterdir() if p.is_dir()), key=str)
               if kaggle_input.exists() else [])
    candidates = [
        Path("/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction"),
        Path("/kaggle/input/parkinsons-freezing-gait-prediction"),
        *mounted,
        *(p / "data" for p in mounted),
        Path.cwd() / "data",
        Path.cwd().parent / "data",
    ]
    if kaggle_input.exists():
        for defog_dir in kaggle_input.rglob("defog"):
            if (defog_dir.is_dir() and defog_dir.parent.name == "train"
                    and (defog_dir.parent / "tdcsfog").exists()):
                candidates.insert(0, defog_dir.parent.parent)
    for base in candidates:
        if (base / "train" / "defog").exists():
            return base, base, base
        if (base / "raw" / "train" / "defog").exists():
            return base, base / "raw", base / "metadata"
    visible = [str(p) for p in mounted]
    raise FileNotFoundError(
        "Could not find train/defog in any Kaggle input. "
        f"Mounted input folders: {visible or '[none]'}. "
        "Attach the Parkinson's Freezing of Gait Prediction competition data."
    )

DATA_ROOT, RAW_ROOT, META_ROOT = locate_layout()
print(f"[FoG] Competition data found at {DATA_ROOT}.", flush=True)
PROJECT_ROOT = DATA_ROOT.parent if (DATA_ROOT / "raw").exists() else Path.cwd()
if Path("/kaggle/working").exists():
    ARTIFACT_ROOT = Path("/kaggle/working/fog-evidence-weighted") / RUN_PROFILE
else:
    ARTIFACT_ROOT = PROJECT_ROOT / "results" / "kaggle_fog_evidence_weighted" / RUN_PROFILE
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"profile={RUN_PROFILE}  neural={RUN_NEURAL}  submission={RUN_FINAL_SUBMISSION}")
print(f"data={DATA_ROOT.resolve()}")
print(f"artifacts={ARTIFACT_ROOT.resolve()}")
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | pandas {pd.__version__}")
print(f"scikit-learn {sklearn.__version__} | TensorFlow {tf.__version__} | GPU={bool(gpus)}")
if RUN_PROFILE != "smoke" and RUN_NEURAL and not gpus:
    raise RuntimeError("The competition profiles require a GPU; use smoke on CPU.")
display(pd.DataFrame([asdict(p) for p in PROFILES.values()]).set_index("name"))

### Method promotion policy

The defaults above are evidence-backed.  Costly or inconsistent ideas
remain opt-in: focal loss, heavy stretch/drift/channel ablation, hard
pseudo-labels, Daily Living self-supervision, residual no-activity GRU,
and ResNet.  Promote one only if it gains at least **0.002** pooled OOF
mAP, does not lose more than **0.005** in either domain, and improves at
least three of five folds.  This prevents a large model zoo from
overfitting one validation split.

## 2. Validate and index the competition data

In [ ]:
def read_csv_required(path: Path, required: set[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing)}")
    return frame

EVENTS = read_csv_required(META_ROOT / "events.csv", {"Id", "Init", "Completion", "Type"})
TASKS = read_csv_required(META_ROOT / "tasks.csv", {"Id", "Begin", "End", "Task"})
SUBJECTS = read_csv_required(META_ROOT / "subjects.csv", {"Subject", "Visit"})
METADATA = {
    domain: read_csv_required(META_ROOT / spec["metadata"], {"Id", "Subject", "Visit"})
    for domain, spec in DOMAIN.items()
}

@dataclass(frozen=True)
class Record:
    domain: str
    recording_id: str
    subject: str
    visit: float
    medication: str
    test: float
    path: Path
    is_train: bool
    n_rows: int
    label_mode: str = "typed"  # typed | event_only | test

def fast_row_count(path: Path) -> int:
    # Time is contiguous 0..N-1. Read only the tail instead of scanning
    # ~1.35 GB of mounted CSVs during startup; full load audits it later.
    with path.open("rb") as handle:
        handle.seek(0, 2)
        size = handle.tell()
        handle.seek(max(0, size - 65536))
        tail = handle.read().decode("utf-8", errors="strict")
    lines = [line for line in tail.splitlines() if line.strip()]
    if not lines:
        raise ValueError(f"Empty CSV: {path}")
    try:
        return int(lines[-1].split(",", 1)[0]) + 1
    except Exception as exc:
        raise ValueError(f"Cannot infer row count from {path}") from exc

def build_records(domain: str, split: str) -> list[Record]:
    folder = RAW_ROOT / split / domain
    if not folder.exists():
        return []
    meta = METADATA[domain].copy()
    meta["Id"] = meta["Id"].astype(str)
    meta = meta.drop_duplicates("Id").set_index("Id")
    out = []
    for path in sorted(folder.glob("*.csv")):
        rid = path.stem
        row = meta.loc[rid] if rid in meta.index else pd.Series(dtype=object)
        out.append(Record(
            domain=domain,
            recording_id=rid,
            subject=str(row.get("Subject", f"unknown:{rid}")),
            visit=float(row.get("Visit", np.nan)),
            medication=str(row.get("Medication", "unknown")),
            test=float(row.get("Test", np.nan)),
            path=path,
            is_train=(split == "train"),
            n_rows=fast_row_count(path),
            label_mode="typed" if split == "train" else "test",
        ))
    return out

TRAIN_RECORDS = {d: build_records(d, "train") for d in DOMAIN}
TEST_RECORDS = {d: build_records(d, "test") for d in DOMAIN}

def build_event_only_records() -> list[Record]:
    candidates = [RAW_ROOT / "unlabeled", RAW_ROOT / "notype",
                  RAW_ROOT / "train" / "notype", RAW_ROOT / "train" / "unlabeled"]
    folder = next((p for p in candidates if p.exists()), None)
    if folder is None:
        return []
    meta = METADATA["defog"].copy()
    meta["Id"] = meta["Id"].astype(str)
    meta = meta.drop_duplicates("Id").set_index("Id")
    out = []
    for path in sorted(folder.glob("*.csv")):
        header = set(pd.read_csv(path, nrows=0).columns)
        if "Event" not in header:
            continue
        rid = path.stem
        row = meta.loc[rid] if rid in meta.index else pd.Series(dtype=object)
        out.append(Record(
            "defog", rid, str(row.get("Subject", f"unknown:{rid}")),
            float(row.get("Visit", np.nan)), str(row.get("Medication", "unknown")),
            float(row.get("Test", np.nan)), path, True, fast_row_count(path), "event_only"
        ))
    return out

AUX_RECORDS = {"defog": build_event_only_records(), "tdcsfog": []}

# The public code-competition example can duplicate a train recording.
# Exclude both recording and subject from fitting; hidden test remains safe.
overlap_audit = []
for domain in DOMAIN:
    test_ids = {r.recording_id for r in TEST_RECORDS[domain]}
    test_subjects = {r.subject for r in TEST_RECORDS[domain] if not r.subject.startswith("unknown:")}
    kept = []
    for r in TRAIN_RECORDS[domain]:
        reason = None
        if r.recording_id in test_ids:
            reason = "recording_id"
        elif r.subject in test_subjects:
            reason = "subject"
        if reason:
            overlap_audit.append({"domain": domain, "recording_id": r.recording_id,
                                  "subject": r.subject, "reason": reason})
        else:
            kept.append(r)
    TRAIN_RECORDS[domain] = kept
    AUX_RECORDS[domain] = [r for r in AUX_RECORDS[domain]
                           if r.recording_id not in test_ids and r.subject not in test_subjects]

counts = []
for domain in DOMAIN:
    counts.append({
        "domain": domain,
        "train_records": len(TRAIN_RECORDS[domain]),
        "train_subjects": len({r.subject for r in TRAIN_RECORDS[domain]}),
        "test_records": len(TEST_RECORDS[domain]),
        "test_rows": sum(r.n_rows for r in TEST_RECORDS[domain]),
        "event_only_aux_records": len(AUX_RECORDS[domain]),
    })
display(pd.DataFrame(counts))
display(pd.DataFrame(overlap_audit))

if not all(TEST_RECORDS.values()):
    raise RuntimeError("Both test domains are required for submission construction.")
print("Blank DeFOG test-event types:", int(EVENTS["Type"].isna().sum()))
print("Task-covered recording IDs:", TASKS["Id"].astype(str).nunique())

## 3. Frozen subject folds, lazy records, and exact native metric

In [ ]:
def event_duration_vector(record: Record) -> np.ndarray:
    rows = EVENTS[(EVENTS["Id"].astype(str) == record.recording_id) & EVENTS["Type"].notna()]
    out = np.zeros(3, dtype=np.float64)
    for name, grp in rows.groupby("Type"):
        if name in TARGET_TO_INDEX:
            out[TARGET_TO_INDEX[name]] = np.maximum(
                grp["Completion"].to_numpy(float) - grp["Init"].to_numpy(float), 0
            ).sum()
    return out

def smoke_subset(records: list[Record], cap: int | None) -> list[Record]:
    if cap is None or len(records) <= cap:
        return records
    # Select subjects, not arbitrary rows. Rare-event coverage is ranked first.
    by_subject: dict[str, list[Record]] = {}
    for rec in records:
        by_subject.setdefault(rec.subject, []).append(rec)
    totals = np.sum([event_duration_vector(r) for r in records], axis=0) + 1e-6
    ranked = []
    for subject, group in by_subject.items():
        vec = np.sum([event_duration_vector(r) for r in group], axis=0)
        rarity = float(np.sum((vec > 0) / np.sqrt(totals)) + np.sum(vec / totals))
        ranked.append((rarity, subject, group))
    ranked.sort(key=lambda item: (-item[0], item[1]))
    # Smoke mode values subject coverage over multiple visits from one person.
    chosen: list[Record] = []
    for _, _, group in ranked[:cap]:
        representative = max(group, key=lambda r: float(event_duration_vector(r).sum()))
        chosen.append(representative)
    if len(chosen) < min(cap, len(records)):
        used = {r.recording_id for r in chosen}
        chosen.extend([r for r in records if r.recording_id not in used][: cap - len(chosen)])
    return sorted(chosen[:cap], key=lambda r: r.recording_id)

if CFG.max_records_per_domain is not None:
    TRAIN_RECORDS = {
        d: smoke_subset(records, CFG.max_records_per_domain)
        for d, records in TRAIN_RECORDS.items()
    }
    AUX_RECORDS = {
        d: records[:CFG.max_records_per_domain]
        for d, records in AUX_RECORDS.items()
    }

def assign_subject_folds(records: list[Record], n_folds: int, seed: int) -> dict[str, int]:
    by_subject: dict[str, list[Record]] = {}
    for rec in records:
        by_subject.setdefault(rec.subject, []).append(rec)
    if len(by_subject) < n_folds:
        raise ValueError(f"Need at least {n_folds} subjects, found {len(by_subject)}")

    subject_rows = []
    for subject, group in by_subject.items():
        durations = np.sum([event_duration_vector(r) for r in group], axis=0)
        subject_rows.append((subject, durations, len(group), sum(r.n_rows for r in group)))
    total_duration = np.sum([x[1] for x in subject_rows], axis=0) + 1e-6
    rng = np.random.default_rng(seed)
    jitter = {row[0]: rng.uniform(0, 1e-6) for row in subject_rows}
    subject_rows.sort(
        key=lambda x: (-(np.sum((x[1] > 0) / np.sqrt(total_duration))
                         + np.sum(x[1] / total_duration)), jitter[x[0]])
    )

    fold_duration = np.zeros((n_folds, 3), dtype=float)
    fold_records = np.zeros(n_folds, dtype=float)
    fold_rows = np.zeros(n_folds, dtype=float)
    fold_subjects = np.zeros(n_folds, dtype=float)
    assignment: dict[str, int] = {}
    duration_scale = total_duration / n_folds
    records_scale = max(len(records) / n_folds, 1)
    rows_scale = max(sum(r.n_rows for r in records) / n_folds, 1)

    for subject, durations, n_records, n_rows in subject_rows:
        costs = []
        for fold in range(n_folds):
            d = fold_duration.copy(); d[fold] += durations
            nr = fold_records.copy(); nr[fold] += n_records
            rw = fold_rows.copy(); rw[fold] += n_rows
            ns = fold_subjects.copy(); ns[fold] += 1
            cost = (
                np.square(d / np.maximum(duration_scale, 1e-6) - 1).mean()
                + 0.12 * np.square(nr / records_scale - 1).mean()
                + 0.08 * np.square(rw / rows_scale - 1).mean()
                + 0.05 * np.var(ns)
            )
            costs.append(float(cost))
        chosen = int(np.argmin(costs))
        assignment[subject] = chosen
        fold_duration[chosen] += durations
        fold_records[chosen] += n_records
        fold_rows[chosen] += n_rows
        fold_subjects[chosen] += 1
    return assignment

def assign_defog_event_folds(records: list[Record], n_folds: int, seed: int) -> dict[str, int]:
    rec_by_id = {r.recording_id: r for r in records}
    typed = EVENTS[
        EVENTS["Id"].astype(str).isin(rec_by_id) & EVENTS["Type"].isin(TARGETS)
    ].copy()
    typed["Subject"] = typed["Id"].astype(str).map(
        lambda rid: rec_by_id[rid].subject
    )
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    assignment: dict[str, int] = {}
    y = pd.Categorical(typed["Type"], categories=TARGETS).codes
    for fold, (_, val_index) in enumerate(splitter.split(np.zeros(len(typed)), y, typed["Subject"])):
        for subject in typed.iloc[val_index]["Subject"].unique():
            assignment[str(subject)] = fold
    # Subjects with no typed event still belong to a deterministic fold.
    for rec in records:
        if rec.subject not in assignment:
            stable = int(hashlib.sha256(rec.subject.encode()).hexdigest()[:8], 16)
            assignment[rec.subject] = stable % n_folds
    return assignment

FOLD_BY_SUBJECT = {
    domain: (assign_defog_event_folds(records, CFG.n_folds, SEED)
             if domain == "defog" else
             assign_subject_folds(records, CFG.n_folds, SEED + i * 101))
    for i, (domain, records) in enumerate(TRAIN_RECORDS.items())
}
FOLD_BY_RECORD = {
    domain: {r.recording_id: FOLD_BY_SUBJECT[domain][r.subject] for r in records}
    for domain, records in TRAIN_RECORDS.items()
}
AUX_FOLD_BY_RECORD = {}
for domain, records in AUX_RECORDS.items():
    AUX_FOLD_BY_RECORD[domain] = {}
    for rec in records:
        if rec.subject in FOLD_BY_SUBJECT[domain]:
            fold = FOLD_BY_SUBJECT[domain][rec.subject]
        else:
            stable = int(hashlib.sha256(rec.subject.encode()).hexdigest()[:8], 16)
            fold = stable % CFG.n_folds
        AUX_FOLD_BY_RECORD[domain][rec.recording_id] = fold

fold_rows = []
for domain, records in TRAIN_RECORDS.items():
    for rec in records:
        fold_rows.append({
            "domain": domain, "recording_id": rec.recording_id,
            "subject": rec.subject, "fold": FOLD_BY_RECORD[domain][rec.recording_id],
            "n_rows": rec.n_rows,
            **{f"seconds_{t}": event_duration_vector(rec)[i] for i, t in enumerate(TARGETS)},
        })
FOLD_FRAME = pd.DataFrame(fold_rows).sort_values(["domain", "fold", "subject", "recording_id"])
FOLD_FRAME.to_csv(ARTIFACT_ROOT / "immutable_subject_folds.csv", index=False)
display(FOLD_FRAME.groupby(["domain", "fold"])[["subject", "recording_id", "n_rows"]]
        .agg({"subject": "nunique", "recording_id": "count", "n_rows": "sum"}))
display(FOLD_FRAME.groupby(["domain", "fold"])[[f"seconds_{t}" for t in TARGETS]].sum())

for domain in DOMAIN:
    for fold in range(CFG.n_folds):
        train_subjects = {r.subject for r in TRAIN_RECORDS[domain]
                          if FOLD_BY_RECORD[domain][r.recording_id] != fold}
        val_subjects = {r.subject for r in TRAIN_RECORDS[domain]
                        if FOLD_BY_RECORD[domain][r.recording_id] == fold}
        assert train_subjects.isdisjoint(val_subjects)

class RecordCache:
    def __init__(self, max_items: int = 8):
        self.max_items = max_items
        self.data: OrderedDict[tuple[str, str, str], dict] = OrderedDict()

    def _directory(self, rec: Record) -> Path:
        stat = rec.path.stat()
        digest = hashlib.sha256()
        digest.update(marshal.dumps(type(self).get.__code__))
        digest.update(
            f"{rec.path.name}:{stat.st_size}:{stat.st_mtime_ns}:{rec.label_mode}:"
            f"{np.__version__}:{pd.__version__}".encode()
        )
        return (ARTIFACT_ROOT / "prepared" / rec.domain / rec.recording_id /
                digest.hexdigest()[:16])

    def get(self, rec: Record) -> dict:
        key = (rec.domain, rec.recording_id, rec.label_mode)
        if key in self.data:
            self.data.move_to_end(key)
            return self.data[key]
        directory = self._directory(rec)
        fields = ("raw", "labels", "score_mask", "event_label", "target_mask4", "time")
        paths = {name: directory / f"{name}.npy" for name in fields}
        if not all(path.exists() for path in paths.values()):
            directory.mkdir(parents=True, exist_ok=True)
            header = set(pd.read_csv(rec.path, nrows=0).columns)
            usecols = ["Time", *AXES]
            if rec.label_mode == "typed":
                usecols += TARGETS
                if rec.domain == "defog":
                    usecols += ["Valid", "Task"]
            elif rec.label_mode == "event_only":
                usecols += ["Event"]
                usecols += [c for c in ("Valid", "Task") if c in header]
            frame = pd.read_csv(rec.path, usecols=usecols)
            raw = frame[AXES].to_numpy(np.float32)
            if not np.isfinite(raw).all():
                med = np.nanmedian(raw, axis=0)
                bad = ~np.isfinite(raw)
                raw[bad] = np.take(med, np.where(bad)[1])
            if rec.label_mode == "typed":
                labels = frame[TARGETS].to_numpy(np.float32)
                if rec.domain == "defog":
                    score_mask = (frame["Valid"].astype(bool) & frame["Task"].astype(bool)).to_numpy()
                else:
                    score_mask = np.ones(len(frame), dtype=bool)
                event_label = labels.max(axis=1).astype(np.float32)
                target_mask4 = np.repeat(score_mask[:, None], 4, axis=1).astype(np.float32)
            elif rec.label_mode == "event_only":
                labels = np.zeros((len(frame), 3), np.float32)
                score_mask = np.ones(len(frame), dtype=bool)
                if "Valid" in frame:
                    score_mask &= frame["Valid"].astype(bool).to_numpy()
                if "Task" in frame:
                    score_mask &= frame["Task"].astype(bool).to_numpy()
                event_label = frame["Event"].to_numpy(np.float32)
                target_mask4 = np.zeros((len(frame), 4), np.float32)
                target_mask4[:, 3] = score_mask.astype(np.float32)
            else:
                labels = np.zeros((len(frame), 3), np.float32)
                score_mask = np.ones(len(frame), dtype=bool)
                event_label = np.zeros(len(frame), np.float32)
                target_mask4 = np.zeros((len(frame), 4), np.float32)
            arrays = {"raw": raw, "labels": labels, "score_mask": score_mask,
                      "event_label": event_label, "target_mask4": target_mask4,
                      "time": frame["Time"].to_numpy(np.int64)}
            if len(frame) != rec.n_rows:
                raise ValueError(f"Row-count audit failed for {rec.recording_id}")
            for name, array in arrays.items():
                temporary = directory / f"{name}.tmp.npy"
                np.save(temporary, array, allow_pickle=False)
                os.replace(temporary, paths[name])
            del frame, arrays
        item = {name: np.load(path, mmap_mode="r", allow_pickle=False)
                for name, path in paths.items()}
        self.data[key] = item
        self.data.move_to_end(key)
        while len(self.data) > self.max_items:
            self.data.popitem(last=False)
        return item

    def clear(self):
        self.data.clear()

CACHE = RecordCache(max_items=6 if RUN_PROFILE != "smoke" else 12)

def exact_ap(records: list[Record], predictions: dict[str, np.ndarray]) -> dict[str, float]:
    ys, ps = [], []
    for rec in records:
        item = CACHE.get(rec)
        pred = predictions[rec.recording_id]
        if pred.shape != item["labels"].shape:
            raise ValueError(f"Prediction shape mismatch for {rec.recording_id}: {pred.shape}")
        mask = item["score_mask"]
        ys.append(item["labels"][mask])
        ps.append(np.clip(pred[mask], EPS, 1 - EPS))
    y = np.concatenate(ys)
    p = np.concatenate(ps)
    out = {}
    for i, name in enumerate(TARGETS):
        out[name] = (float(average_precision_score(y[:, i], p[:, i]))
                     if np.unique(y[:, i]).size > 1 else np.nan)
    out["mAP"] = float(np.nanmean([out[t] for t in TARGETS]))
    return out

print("Fold manifest hash:", hashlib.sha256(
    FOLD_FRAME.to_csv(index=False).encode("utf-8")
).hexdigest()[:16])

The fold table is the first audit artifact.  A fold with no rare-class
duration is reported rather than silently hidden; `StartHesitation` is
especially unstable in DeFOG, so pooled AP and fold dispersion must both
be read.

## 4. DeFOG event/task/clinical specialist (the highest-value branch)

In [ ]:
def interval_indices(init_s: float, completion_s: float, hz: int, n: int) -> tuple[int, int]:
    # Verified against train labels. Stop is exclusive. Clamp before slicing:
    # a negative Python index would otherwise paint from the array tail.
    start = int(math.ceil(float(init_s) * hz))
    stop = int(math.floor(float(completion_s) * hz)) + 1
    start = min(max(start, 0), n)
    stop = min(max(stop, start), n)
    return start, stop

def task_at(recording_id: str, midpoint_s: float) -> tuple[str, float, float, float]:
    rows = TASKS[TASKS["Id"].astype(str) == recording_id]
    if rows.empty:
        return "Unknown", np.nan, np.nan, np.nan
    inside = rows[(rows["Begin"] <= midpoint_s) & (rows["End"] > midpoint_s)]
    if inside.empty:
        distance = np.minimum(abs(rows["Begin"] - midpoint_s), abs(rows["End"] - midpoint_s))
        row = rows.iloc[int(np.argmin(distance.to_numpy()))]
        return "NO_TASK", float(row["Begin"]), float(row["End"]), np.nan
    row = inside.iloc[0]
    duration = max(float(row["End"] - row["Begin"]), 1e-6)
    position = (midpoint_s - float(row["Begin"])) / duration
    return str(row["Task"]), float(row["Begin"]), float(row["End"]), float(position)

def subject_features(rec: Record) -> dict[str, object]:
    rows = SUBJECTS[SUBJECTS["Subject"].astype(str) == rec.subject]
    if not rows.empty and np.isfinite(rec.visit):
        exact = rows[pd.to_numeric(rows["Visit"], errors="coerce") == rec.visit]
        if not exact.empty:
            rows = exact
    row = rows.iloc[0] if not rows.empty else pd.Series(dtype=object)
    return {
        "Visit": rec.visit,
        "Medication": rec.medication,
        "Test": rec.test,
        "Sex": str(row.get("Sex", "unknown")),
        "Age": pd.to_numeric(row.get("Age", np.nan), errors="coerce"),
        "YearsSinceDx": pd.to_numeric(row.get("YearsSinceDx", np.nan), errors="coerce"),
        "UPDRSIII_On": pd.to_numeric(row.get("UPDRSIII_On", np.nan), errors="coerce"),
        "UPDRSIII_Off": pd.to_numeric(row.get("UPDRSIII_Off", np.nan), errors="coerce"),
        "NFOGQ": pd.to_numeric(row.get("NFOGQ", np.nan), errors="coerce"),
    }

def safe_summary(x: np.ndarray, prefix: str) -> dict[str, float]:
    if x.size == 0:
        return {f"{prefix}_{name}": np.nan for name in
                ("mean", "std", "rms", "q10", "q90", "range", "jerk_rms")}
    x = np.asarray(x, np.float64)
    d = np.diff(x)
    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std": float(np.std(x)),
        f"{prefix}_rms": float(np.sqrt(np.mean(x * x))),
        f"{prefix}_q10": float(np.quantile(x, .10)),
        f"{prefix}_q90": float(np.quantile(x, .90)),
        f"{prefix}_range": float(np.ptp(x)),
        f"{prefix}_jerk_rms": float(np.sqrt(np.mean(d * d))) if d.size else 0.0,
    }

def spectral_summary(x: np.ndarray, hz: int, prefix: str) -> dict[str, float]:
    if len(x) < 16:
        return {f"{prefix}_bp_{lo:g}_{hi:g}": np.nan
                for lo, hi in ((0.5, 3), (3, 8), (8, 20))}
    stride = max(1, len(x) // 4096)
    f, p = periodogram(np.asarray(x[::stride], np.float64), fs=hz / stride)
    total = float(np.trapz(p, f)) + 1e-12
    out = {}
    for lo, hi in ((0.5, 3), (3, 8), (8, 20)):
        band = (f >= lo) & (f < hi)
        out[f"{prefix}_bp_{lo:g}_{hi:g}"] = (
            float(np.trapz(p[band], f[band]) / total) if band.sum() >= 2 else 0.0
        )
    return out

def rich_region_summary(xyz: np.ndarray, hz: int, prefix: str,
                        spectral: bool) -> dict[str, float]:
    out: dict[str, float] = {}
    if len(xyz) == 0:
        return out
    magnitude = np.linalg.norm(xyz, axis=1, keepdims=True)
    channels = np.concatenate([xyz, magnitude], axis=1)
    names = [*AXES, "Magnitude"]
    for i, name in enumerate(names):
        values = np.asarray(channels[:, i], np.float64)
        diff = np.diff(values) * hz
        for quantile in (.10, .25, .50, .75, .90):
            out[f"{prefix}_{name}_q{int(quantile*100):02d}"] = float(np.quantile(values, quantile))
        out.update({
            f"{prefix}_{name}_mean": float(np.mean(values)),
            f"{prefix}_{name}_std": float(np.std(values)),
            f"{prefix}_{name}_min": float(np.min(values)),
            f"{prefix}_{name}_max": float(np.max(values)),
            f"{prefix}_{name}_rms": float(np.sqrt(np.mean(values * values))),
            f"{prefix}_{name}_jerk_abs_mean": float(np.mean(abs(diff))) if len(diff) else 0.0,
            f"{prefix}_{name}_jerk_std": float(np.std(diff)) if len(diff) else 0.0,
        })
        if spectral and len(values) >= 16:
            f, p = welch(values, fs=hz, nperseg=min(256, len(values)),
                         noverlap=0, detrend="constant")
            powers = {}
            for lo, hi in ((.5, 3), (3, 8), (8, 20)):
                band = (f >= lo) & (f < hi)
                power = float(np.trapz(p[band], f[band])) if band.sum() >= 2 else 0.0
                powers[(lo, hi)] = power
                out[f"{prefix}_{name}_logpower_{lo:g}_{hi:g}"] = float(np.log1p(power))
            out[f"{prefix}_{name}_log_freeze_ratio"] = float(
                np.log1p(powers[(3, 8)] / max(powers[(.5, 3)], 1e-12))
            )
    for left, right in ((0, 1), (0, 2), (1, 2)):
        corr = np.corrcoef(channels[:, left], channels[:, right])[0, 1]
        out[f"{prefix}_corr_{AXES[left]}_{AXES[right]}"] = float(corr) if np.isfinite(corr) else 0.0
    return out

def event_feature_table(records: list[Record], typed: bool) -> pd.DataFrame:
    rec_by_id = {r.recording_id: r for r in records}
    rows = EVENTS[EVENTS["Id"].astype(str).isin(rec_by_id)].copy()
    rows = rows[rows["Type"].notna() if typed else rows["Type"].isna()]
    output = []
    for source_index, event in rows.iterrows():
        rid = str(event["Id"])
        rec = rec_by_id[rid]
        item = CACHE.get(rec)
        n = len(item["raw"])
        start, stop = interval_indices(event["Init"], event["Completion"], DOMAIN[rec.domain]["hz"], n)
        midpoint = 0.5 * (float(event["Init"]) + float(event["Completion"]))
        task, task_begin, task_end, task_position = task_at(rid, midpoint)
        segment = item["raw"][start:stop]
        context_pad = 2 * DOMAIN[rec.domain]["hz"]
        pre = item["raw"][max(0, start - context_pad):start]
        post = item["raw"][stop:min(n, stop + context_pad)]
        context = item["raw"][max(0, start - context_pad):min(n, stop + context_pad)]
        scored_rows = int(item["score_mask"][start:stop].sum()) if rec.is_train else stop - start
        row = {
            "event_key": f"{rid}:{source_index}",
            "recording_id": rid,
            "subject": rec.subject,
            "fold": FOLD_BY_RECORD.get("defog", {}).get(rid, -1),
            "start": start,
            "stop": stop,
            "Init": float(event["Init"]),
            "Completion": float(event["Completion"]),
            "duration": max(float(event["Completion"] - event["Init"]), 0),
            "EventMidpointSeconds": midpoint,
            "midpoint_fraction": midpoint / max(n / DOMAIN[rec.domain]["hz"], 1e-6),
            "start_fraction": start / max(n, 1),
            "stop_fraction": stop / max(n, 1),
            "TaskName": task,
            "task_begin": task_begin,
            "task_end": task_end,
            "task_position": task_position,
            "seconds_from_task_start": midpoint - task_begin if np.isfinite(task_begin) else np.nan,
            "seconds_to_task_end": task_end - midpoint if np.isfinite(task_end) else np.nan,
            "scored_rows": scored_rows,
            "Type": event["Type"] if typed else np.nan,
        }
        row.update(subject_features(rec))
        base_task = task.split("-")[0]
        if task.startswith("Turning"):
            task_family = "Turning"
        elif task.startswith("TUG"):
            task_family = "TUG"
        elif task.startswith("Hotspot"):
            task_family = "Hotspot"
        elif task.startswith("4MW"):
            task_family = "4MW"
        elif task.startswith("MB"):
            task_family = "MB"
        elif task.startswith("Rest"):
            task_family = "Rest"
        else:
            task_family = "Other"
        if task.endswith("-DT"):
            task_modifier = "dual_task"
        elif task.endswith("-ST"):
            task_modifier = "single_task"
        elif task.endswith("-C"):
            task_modifier = "cued"
        else:
            task_modifier = "plain"
        row["TaskFamily"] = task_family
        row["TaskModifier"] = task_modifier
        for i, axis in enumerate(AXES):
            values = segment[:, i] if len(segment) else np.array([])
            context_values = context[:, i] if len(context) else np.array([])
            row.update(safe_summary(values, f"event_{axis}"))
            row.update(safe_summary(context_values, f"context_{axis}"))
            row.update(spectral_summary(values, DOMAIN[rec.domain]["hz"], f"event_{axis}"))
        if len(segment):
            row.update(safe_summary(np.linalg.norm(segment, axis=1), "event_magnitude"))
        row.update(rich_region_summary(pre, DOMAIN[rec.domain]["hz"], "pre2", False))
        row.update(rich_region_summary(segment, DOMAIN[rec.domain]["hz"], "event_rich", True))
        row.update(rich_region_summary(post, DOMAIN[rec.domain]["hz"], "post2", False))
        row.update(rich_region_summary(context, DOMAIN[rec.domain]["hz"], "context_rich", True))
        output.append(row)
    return pd.DataFrame(output)

def event_feature_fingerprint() -> str:
    digest = hashlib.sha256()
    for fn in (interval_indices, task_at, subject_features, safe_summary,
               spectral_summary, rich_region_summary, event_feature_table):
        digest.update(marshal.dumps(fn.__code__))
    paths = [r.path for r in TRAIN_RECORDS["defog"] + TEST_RECORDS["defog"]]
    paths += [META_ROOT / name for name in ("events.csv", "tasks.csv", "subjects.csv",
                                            "defog_metadata.csv")]
    for path in sorted(paths, key=str):
        stat = path.stat()
        digest.update(f"{path.name}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    return digest.hexdigest()

event_train_path = ARTIFACT_ROOT / "defog_event_features_train.csv"
event_test_path = ARTIFACT_ROOT / "defog_event_features_test.csv"
event_cache_manifest = ARTIFACT_ROOT / "defog_event_feature_cache.json"
feature_fingerprint = event_feature_fingerprint()
cache_ok = False
if event_train_path.exists() and event_test_path.exists() and event_cache_manifest.exists():
    cache_ok = json.loads(event_cache_manifest.read_text(encoding="utf-8")).get("fingerprint") == feature_fingerprint
if cache_ok:
    DEFOG_EVENT_TRAIN = pd.read_csv(event_train_path)
    DEFOG_EVENT_TEST = pd.read_csv(event_test_path)
    print("Reused fingerprinted DeFOG event-feature cache.")
else:
    DEFOG_EVENT_TRAIN = event_feature_table(TRAIN_RECORDS["defog"], typed=True)
    DEFOG_EVENT_TEST = event_feature_table(TEST_RECORDS["defog"], typed=False)
    DEFOG_EVENT_TRAIN.to_csv(event_train_path, index=False)
    DEFOG_EVENT_TEST.to_csv(event_test_path, index=False)
    atomic_payload = {"fingerprint": feature_fingerprint, "created_utc": pd.Timestamp.utcnow().isoformat()}
    temp_manifest = event_cache_manifest.with_suffix(".tmp.json")
    temp_manifest.write_text(json.dumps(atomic_payload, indent=2), encoding="utf-8")
    os.replace(temp_manifest, event_cache_manifest)

ZERO_SCORE_EVENTS = DEFOG_EVENT_TRAIN["scored_rows"] <= 0
if ZERO_SCORE_EVENTS.any():
    print(f"Excluding {int(ZERO_SCORE_EVENTS.sum())} typed events with zero Valid&Task overlap.")
DEFOG_EVENT_TRAIN_FIT = DEFOG_EVENT_TRAIN.loc[~ZERO_SCORE_EVENTS].reset_index(drop=True)
print(f"Event features: train={len(DEFOG_EVENT_TRAIN_FIT):,}, test={len(DEFOG_EVENT_TEST):,}")

In [ ]:
EVENT_CATEGORICAL = ["TaskName", "TaskFamily", "TaskModifier", "Medication", "Sex"]
EVENT_EXCLUDE = {
    "event_key", "recording_id", "subject", "fold", "start", "stop", "Type",
    "scored_rows", "Kinetic",  # populated in train, blank test: never a feature
}

def event_feature_columns(frame: pd.DataFrame):
    numeric = [c for c in frame.columns
               if c not in EVENT_EXCLUDE and c not in EVENT_CATEGORICAL
               and pd.to_numeric(frame[c], errors="coerce").notna().any()]
    return EVENT_CATEGORICAL, numeric

def make_event_classifier(frame: pd.DataFrame, seed: int) -> Pipeline:
    categorical, numeric = event_feature_columns(frame)
    prep = ColumnTransformer([
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
    ], remainder="drop")
    model = HistGradientBoostingClassifier(
        learning_rate=0.04, max_iter=300, max_leaf_nodes=15,
        min_samples_leaf=12, l2_regularization=3.0,
        early_stopping=False, random_state=seed,
    )
    return Pipeline([("features", prep), ("model", model)])

def event_sample_weights(frame: pd.DataFrame) -> np.ndarray:
    counts = frame["Type"].value_counts()
    class_factor = frame["Type"].map(
        lambda x: math.sqrt(len(frame) / (3 * max(counts[x], 1)))
    ).to_numpy(float)
    duration = np.maximum(frame["duration"].to_numpy(float), .01)
    return class_factor * np.sqrt(duration)

def aligned_proba(clf: Pipeline, frame: pd.DataFrame) -> np.ndarray:
    raw = clf.predict_proba(frame)
    classes = list(clf.named_steps["model"].classes_)
    out = np.full((len(frame), 3), EPS, np.float32)
    for j, name in enumerate(classes):
        if name in TARGET_TO_INDEX:
            out[:, TARGET_TO_INDEX[name]] = raw[:, j]
    out /= np.maximum(out.sum(axis=1, keepdims=True), EPS)
    return out

def crossfit_event_specialist(train_events: pd.DataFrame, test_events: pd.DataFrame):
    oof = np.full((len(train_events), 3), np.nan, np.float32)
    fold_info = []
    for fold in range(CFG.n_folds):
        tr = train_events[train_events["fold"] != fold]
        va = train_events[train_events["fold"] == fold]
        if va.empty:
            continue
        if tr["Type"].nunique() < 2:
            priors = tr["Type"].value_counts(normalize=True)
            pred = np.tile([priors.get(t, EPS) for t in TARGETS], (len(va), 1))
        else:
            clf = make_event_classifier(tr, SEED + fold)
            clf.fit(tr, tr["Type"], model__sample_weight=event_sample_weights(tr))
            pred = aligned_proba(clf, va)
        oof[va.index.to_numpy()] = pred
        fold_info.append({"fold": fold, "train_events": len(tr), "validation_events": len(va),
                          **{f"validation_{t}": int((va["Type"] == t).sum()) for t in TARGETS}})
    if np.isnan(oof).any():
        raise RuntimeError("Some event OOF rows were not predicted; finish every frozen fold.")

    if train_events["Type"].nunique() < 2:
        raise RuntimeError("Event specialist needs at least two event types.")
    final = make_event_classifier(train_events, SEED + 999)
    final.fit(train_events, train_events["Type"],
              model__sample_weight=event_sample_weights(train_events))
    test_pred = (aligned_proba(final, test_events)
                 if len(test_events) else np.zeros((0, 3), np.float32))
    return oof, test_pred, final, pd.DataFrame(fold_info)

def task_prior_crossfit(train_events: pd.DataFrame) -> np.ndarray:
    out = np.full((len(train_events), 3), EPS, np.float32)
    for fold in range(CFG.n_folds):
        tr = train_events[train_events["fold"] != fold]
        va = train_events[train_events["fold"] == fold]
        global_count = tr["Type"].value_counts().reindex(TARGETS, fill_value=0).to_numpy(float) + 1
        table = pd.crosstab(tr["TaskName"], tr["Type"]).reindex(columns=TARGETS, fill_value=0)
        for idx, row in va.iterrows():
            counts = (table.loc[row["TaskName"]].to_numpy(float) + 1
                      if row["TaskName"] in table.index else global_count.copy())
            out[idx] = counts / counts.sum()
    return out

def broadcast_events(records: list[Record], events: pd.DataFrame,
                     event_pred: np.ndarray) -> dict[str, np.ndarray]:
    out = {r.recording_id: np.full((r.n_rows, 3), EPS, np.float32) for r in records}
    counts = {r.recording_id: np.zeros(r.n_rows, np.float32) for r in records}
    sums = {r.recording_id: np.zeros((r.n_rows, 3), np.float32) for r in records}
    for pos, (_, event) in enumerate(events.iterrows()):
        rid = event["recording_id"]
        start, stop = int(event["start"]), int(event["stop"])
        if stop <= start or rid not in sums:
            continue
        sums[rid][start:stop] += event_pred[pos]
        counts[rid][start:stop] += 1
    for rid in out:
        mask = counts[rid] > 0
        if mask.any():
            out[rid][mask] = sums[rid][mask] / counts[rid][mask, None]
    return out

EVENT_BASE_OOF, EVENT_BASE_TEST, EVENT_MODEL, EVENT_FOLD_INFO = crossfit_event_specialist(
    DEFOG_EVENT_TRAIN_FIT, DEFOG_EVENT_TEST
)
TASK_PRIOR_OOF = task_prior_crossfit(DEFOG_EVENT_TRAIN_FIT)
event_oof_export = DEFOG_EVENT_TRAIN_FIT[
    ["event_key", "recording_id", "subject", "fold", "Type", "start", "stop", "scored_rows"]
].copy()
for i, name in enumerate(TARGETS):
    event_oof_export[f"task_prior_{name}"] = TASK_PRIOR_OOF[:, i]
    event_oof_export[f"specialist_{name}"] = EVENT_BASE_OOF[:, i]
event_oof_export.to_csv(ARTIFACT_ROOT / "event_specialist_oof_predictions.csv", index=False)
event_test_export = DEFOG_EVENT_TEST[
    ["event_key", "recording_id", "subject", "start", "stop"]
].copy()
for i, name in enumerate(TARGETS):
    event_test_export[f"specialist_{name}"] = EVENT_BASE_TEST[:, i]
event_test_export.to_csv(ARTIFACT_ROOT / "event_specialist_test_predictions.csv", index=False)
EVENT_BASE_OOF_STORE = broadcast_events(
    TRAIN_RECORDS["defog"], DEFOG_EVENT_TRAIN_FIT, EVENT_BASE_OOF
)
EVENT_BASE_TEST_STORE = broadcast_events(
    TEST_RECORDS["defog"], DEFOG_EVENT_TEST, EVENT_BASE_TEST
)
TASK_PRIOR_OOF_STORE = broadcast_events(
    TRAIN_RECORDS["defog"], DEFOG_EVENT_TRAIN_FIT, TASK_PRIOR_OOF
)

EVENT_FOLD_INFO.to_csv(ARTIFACT_ROOT / "event_specialist_fold_counts.csv", index=False)
event_scores = pd.DataFrame([
    {"branch": "task_prior", **exact_ap(TRAIN_RECORDS["defog"], TASK_PRIOR_OOF_STORE)},
    {"branch": "event_task_signal_clinical", **exact_ap(TRAIN_RECORDS["defog"], EVENT_BASE_OOF_STORE)},
])
display(event_scores)
display(EVENT_FOLD_INFO)
display(DEFOG_EVENT_TRAIN_FIT["Type"].value_counts().reindex(TARGETS, fill_value=0).to_frame("events"))

This branch is legal but should be described precisely: the competition
gives the interval, while the model predicts its type.  It does not use
the withheld `Type` or train-only `Kinetic`.  Sensor networks below add
independent event-type evidence and handle TDCS, whose hidden records do
not have the same blank-type interval shortcut.

## 5. Long-context neural branches and event-balanced crops

In [ ]:
@dataclass(frozen=True)
class BranchSpec:
    name: str
    domain: str
    train_length: int
    infer_length: int
    patch_size: int
    output_dim: int
    feature_dim: int
    positive_fraction: float
    full_inverse: bool = False

def branch_spec(domain: str, branch: str) -> BranchSpec:
    if branch == "patch_bilstm":
        patch = 14 if domain == "defog" else 18
        length = patch * CFG.patch_tokens
        return BranchSpec(branch, domain, length, length, patch,
                          4 if domain == "defog" else 3, patch * 3, 0.0)
    if branch in {"engineered_bigru", "wavenet_bigru"}:
        if RUN_PROFILE == "smoke":
            train_length = 1024 if domain == "defog" else 768
            infer_length = train_length * 2
        else:
            train_length = 5000 if domain == "defog" else 1000
            infer_length = 15000 if domain == "defog" else 5000
        feature_dim = 9 if domain == "defog" else 12
        return BranchSpec(branch, domain, train_length, infer_length, 1,
                          4 if domain == "defog" else 3, feature_dim, 0.0)
    if branch == "tdcs_turn_specialist":
        if domain != "tdcsfog":
            raise ValueError("Turn specialist is TDCS-only")
        length = 768 if RUN_PROFILE == "smoke" else 1000
        infer_length = length * 2 if RUN_PROFILE == "smoke" else 5000
        return BranchSpec(branch, domain, length, infer_length, 1, 1, 12, .85, True)
    raise KeyError((domain, branch))

def robust_scale(x: np.ndarray) -> np.ndarray:
    median = np.median(x, axis=0, keepdims=True)
    q25, q75 = np.quantile(x, [.25, .75], axis=0, keepdims=True)
    return ((x - median) / np.maximum(q75 - q25, 1e-5)).astype(np.float32)

def standard_scale(x: np.ndarray) -> np.ndarray:
    mean = np.mean(x, axis=0, keepdims=True)
    std = np.std(x, axis=0, keepdims=True)
    return ((x - mean) / np.maximum(std, 1e-5)).astype(np.float32)

class FeatureCache:
    def __init__(self, max_items=5):
        self.max_items = max_items
        self.data: OrderedDict[tuple[str, str, str], np.ndarray] = OrderedDict()

    def get(self, rec: Record, branch: str) -> np.ndarray:
        family = "patch" if branch == "patch_bilstm" else "engineered"
        key = (family, rec.recording_id, rec.label_mode)
        if key in self.data:
            self.data.move_to_end(key)
            return self.data[key]
        raw = CACHE.get(rec)["raw"]
        digest = hashlib.sha256()
        digest.update(marshal.dumps(type(self).get.__code__))
        digest.update(marshal.dumps(robust_scale.__code__))
        digest.update(marshal.dumps(standard_scale.__code__))
        digest.update(f"{family}:{rec.label_mode}:{rec.path.stat().st_mtime_ns}".encode())
        feature_path = (CACHE._directory(rec) /
                        f"features_{family}_{digest.hexdigest()[:12]}.npy")
        if not feature_path.exists():
            if family == "patch":
                feat = standard_scale(raw)
            else:
                lag = np.vstack([np.zeros((1, 3), np.float32), np.diff(raw, axis=0)])
                lead = np.vstack([np.diff(raw, axis=0), np.zeros((1, 3), np.float32)])
                parts = [raw, lag, lead]
                if rec.domain == "tdcsfog":
                    parts.append(np.cumsum(raw - np.median(raw, axis=0, keepdims=True), axis=0))
                feat = np.concatenate(parts, axis=1)
                feat = robust_scale(feat) if rec.domain == "tdcsfog" else standard_scale(feat)
            temporary = feature_path.with_name(feature_path.stem + ".tmp.npy")
            np.save(temporary, feat.astype(np.float32), allow_pickle=False)
            os.replace(temporary, feature_path)
            del feat
        feat = np.load(feature_path, mmap_mode="r", allow_pickle=False)
        self.data[key] = feat
        self.data.move_to_end(key)
        while len(self.data) > self.max_items:
            self.data.popitem(last=False)
        return feat

    def clear(self):
        self.data.clear()

FEATURE_CACHE = FeatureCache(4 if RUN_PROFILE != "smoke" else 10)

def extract_padded(array: np.ndarray, start: int, length: int,
                   pad_value: float = 0.0, edge: bool = False) -> np.ndarray:
    n = len(array)
    lo, hi = max(start, 0), min(start + length, n)
    core = array[lo:hi]
    before, after = max(0, -start), max(0, start + length - n)
    if before == 0 and after == 0:
        return core.copy()
    if edge and len(core):
        return np.pad(core, [(before, after)] + [(0, 0)] * (array.ndim - 1), mode="edge")
    return np.pad(core, [(before, after)] + [(0, 0)] * (array.ndim - 1),
                  mode="constant", constant_values=pad_value)

def run_centers(binary: np.ndarray) -> np.ndarray:
    binary = np.asarray(binary, bool)
    if not binary.any():
        return np.empty(0, np.int64)
    starts = np.flatnonzero(binary & ~np.r_[False, binary[:-1]])
    stops = np.flatnonzero(binary & ~np.r_[binary[1:], False]) + 1
    return ((starts + stops) // 2).astype(np.int64)

def build_candidate_catalog(records: list[Record]) -> dict:
    catalog = {"positive": {i: [] for i in range(3)}, "event": [], "aux_event": [],
               "negative": [], "uniform": []}
    for rec in records:
        item = CACHE.get(rec)
        mask = item["score_mask"]
        for target in range(3):
            centers = run_centers((item["labels"][:, target] > .5) & mask)
            if len(centers):
                catalog["positive"][target].append((rec, centers))
        event_centers = run_centers((item["event_label"] > .5) & mask)
        if len(event_centers):
            catalog["event"].append((rec, event_centers))
            if rec.label_mode == "event_only":
                catalog["aux_event"].append((rec, event_centers))
        valid_negative = np.flatnonzero(mask & (item["event_label"] < .5))
        if len(valid_negative):
            stride = max(1, len(valid_negative) // 2000)
            catalog["negative"].append((rec, valid_negative[::stride]))
        valid = np.flatnonzero(mask)
        if len(valid) and rec.label_mode == "typed":
            stride = max(1, len(valid) // 4000)
            catalog["uniform"].append((rec, valid[::stride]))
    return catalog

def fixed_validation_catalog(records: list[Record], catalog: dict,
                             per_target: int = 64, negatives: int = 128) -> dict:
    selected = {"positive": {i: [] for i in range(3)}, "event": [],
                "aux_event": [], "negative": [], "uniform": []}
    for target in range(3):
        for rec, centers in catalog["positive"][target]:
            stride = max(1, len(centers) // per_target)
            take = centers[::stride][:per_target]
            if len(take):
                selected["positive"][target].append((rec, take))
                selected["uniform"].append((rec, take))
    remaining = negatives
    for rec, centers in catalog["negative"]:
        if remaining <= 0:
            break
        stride = max(1, len(centers) // min(len(centers), remaining))
        take = centers[::stride][:remaining]
        if len(take):
            selected["negative"].append((rec, take))
            selected["uniform"].append((rec, take))
            remaining -= len(take)
    return selected

CATALOG_CACHE: dict[tuple[str, int, str], dict] = {}

def get_catalog(domain: str, fold: int, split: str) -> tuple[list[Record], dict]:
    if split == "train":
        records = [r for r in TRAIN_RECORDS[domain]
                   if FOLD_BY_RECORD[domain][r.recording_id] != fold]
        records += [r for r in AUX_RECORDS[domain]
                    if AUX_FOLD_BY_RECORD[domain][r.recording_id] != fold]
    else:
        records = [r for r in TRAIN_RECORDS[domain]
                   if FOLD_BY_RECORD[domain][r.recording_id] == fold]
    key = (domain, fold, split)
    if key not in CATALOG_CACHE:
        CATALOG_CACHE[key] = build_candidate_catalog(records)
    return records, CATALOG_CACHE[key]

def choose_center(pool: list[tuple[Record, np.ndarray]], rng: np.random.Generator):
    counts = np.fromiter((len(centers) for _, centers in pool), dtype=np.float64)
    chosen = int(rng.choice(len(pool), p=counts / counts.sum()))
    rec, centers = pool[chosen]
    return rec, int(centers[rng.integers(0, len(centers))])

def augment_raw_window(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    x = x.copy()
    x *= rng.uniform(.92, 1.08, size=(1, 3)).astype(np.float32)
    theta = rng.uniform(-.12, .12)
    rotation = np.array([[1, 0, 0], [0, np.cos(theta), -np.sin(theta)],
                         [0, np.sin(theta), np.cos(theta)]], np.float32)
    x = x @ rotation.T
    if rng.random() < .20:
        drift = np.cumsum(rng.normal(0, 2e-4, size=x.shape), axis=0).astype(np.float32)
        x += drift
    if rng.random() < .06:
        x[:, rng.integers(0, 3)] = 0
    return x

def estimate_positive_weights(records: list[Record], output_dim: int,
                              full_inverse: bool) -> np.ndarray:
    positives = np.zeros(output_dim, np.float64)
    negatives = np.zeros(output_dim, np.float64)
    for rec in records:
        item = CACHE.get(rec)
        mask = item["score_mask"]
        labels = item["labels"]
        if output_dim == 1:
            labels = labels[:, 1:2]
        elif output_dim == 4:
            labels = np.c_[labels, item["event_label"]]
            target_mask = item["target_mask4"].astype(bool)
            positives += ((labels > .5) & target_mask).sum(axis=0)
            negatives += ((labels <= .5) & target_mask).sum(axis=0)
            continue
        positives += (labels[mask] > .5).sum(axis=0)
        negatives += (labels[mask] <= .5).sum(axis=0)
    ratio = negatives / np.maximum(positives, 1)
    weights = ratio if full_inverse else np.sqrt(ratio)
    return np.clip(weights, 1.0, 100.0 if full_inverse else 30.0).astype(np.float32)

In [ ]:
class WindowSequence(tf.keras.utils.Sequence):
    def __init__(self, records: list[Record], catalog: dict, spec: BranchSpec,
                 steps: int, batch_size: int, seed: int,
                 training: bool, augment: bool = False, **kwargs):
        super().__init__(**kwargs)
        self.records = records
        self.catalog = catalog
        self.spec = spec
        self.steps = steps
        self.batch_size = batch_size
        self.seed = seed
        self.training = training
        self.augment = augment and training and spec.name == "patch_bilstm"
        self.epoch = 0
        self.plan: list[tuple[Record, int]] = []
        self.on_epoch_end()

    def __len__(self):
        return self.steps

    def _new_plan(self):
        rng = np.random.default_rng(self.seed + 1009 * self.epoch)
        total = self.steps * self.batch_size
        plan = []
        available_targets = [i for i, rows in self.catalog["positive"].items() if rows]
        if self.spec.name == "tdcs_turn_specialist":
            available_targets = [1] if self.catalog["positive"][1] else []
        for _ in range(total):
            natural = self.spec.positive_fraction <= 0
            use_event_only = (self.spec.output_dim == 4 and self.catalog["aux_event"]
                              and rng.random() < .15)
            use_positive = (not natural and bool(available_targets)
                            and rng.random() < self.spec.positive_fraction)
            if natural and self.catalog["uniform"]:
                if use_event_only:
                    rec, center = choose_center(self.catalog["aux_event"], rng)
                else:
                    rec, center = choose_center(self.catalog["uniform"], rng)
            elif use_event_only:
                rec, center = choose_center(self.catalog["aux_event"], rng)
                center += int(rng.integers(-self.spec.train_length // 5,
                                           self.spec.train_length // 5 + 1))
            elif use_positive:
                target = int(rng.choice(available_targets))
                rec, centers = self.catalog["positive"][target][
                    rng.integers(0, len(self.catalog["positive"][target]))
                ]
                center = int(centers[rng.integers(0, len(centers))])
                center += int(rng.integers(-self.spec.train_length // 5,
                                           self.spec.train_length // 5 + 1))
            elif self.catalog["negative"]:
                rec, center = choose_center(self.catalog["negative"], rng)
            else:
                rec = self.records[rng.integers(0, len(self.records))]
                center = int(rng.integers(0, max(rec.n_rows, 1)))
            start = center - self.spec.train_length // 2
            plan.append((rec, start))
        return plan

    def on_epoch_end(self):
        if not self.training and self.plan:
            return
        self.plan = self._new_plan()
        if self.training:
            self.epoch += 1

    def __getitem__(self, batch_index: int):
        rng = np.random.default_rng(self.seed + self.epoch * 100003 + batch_index)
        xs, ys = [], []
        lo = batch_index * self.batch_size
        for rec, start in self.plan[lo:lo + self.batch_size]:
            item = CACHE.get(rec)
            feat = FEATURE_CACHE.get(rec, self.spec.name)
            x = extract_padded(feat, start, self.spec.train_length, edge=True)
            labels = extract_padded(item["labels"], start, self.spec.train_length)
            if self.augment:
                x = augment_raw_window(x, rng)

            if self.spec.output_dim == 1:
                labels = labels[:, 1:2]
                mask_out = extract_padded(
                    item["score_mask"][:, None],
                    start, self.spec.train_length
                ).astype(np.float32, copy=False)
            elif self.spec.output_dim == 4:
                event_label = extract_padded(
                    item["event_label"][:, None],
                    start, self.spec.train_length
                )
                labels = np.c_[labels, event_label]
                mask_out = extract_padded(
                    item["target_mask4"],
                    start, self.spec.train_length
                )
            else:
                mask = extract_padded(
                    item["score_mask"][:, None],
                    start, self.spec.train_length
                ).astype(np.float32, copy=False)
                mask_out = np.repeat(mask, self.spec.output_dim, axis=1)

            if self.spec.patch_size > 1:
                tokens = self.spec.train_length // self.spec.patch_size
                x = x.reshape(tokens, self.spec.patch_size * x.shape[1])
                labels = labels.reshape(tokens, self.spec.patch_size, self.spec.output_dim)
                patch_mask = mask_out.reshape(tokens, self.spec.patch_size, self.spec.output_dim)
                labels = np.max(labels * patch_mask, axis=1)
                mask_out = np.mean(patch_mask, axis=1)

            xs.append(x.astype(np.float32))
            ys.append(np.concatenate([labels, mask_out], axis=-1).astype(np.float32))
        return np.stack(xs), np.stack(ys)

class MaskedWeightedBCE(tf.keras.losses.Loss):
    def __init__(self, output_dim: int, positive_weights: np.ndarray,
                 gamma: float = 0.0, name="masked_weighted_bce"):
        super().__init__(name=name)
        self.output_dim = output_dim
        self.positive_weights = tf.constant(positive_weights, tf.float32)
        self.gamma = float(gamma)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-6, 1 - 1e-6)
        labels = y_true[..., :self.output_dim]
        mask = y_true[..., self.output_dim:2 * self.output_dim]
        positive = -labels * tf.math.log(y_pred) * self.positive_weights
        negative = -(1 - labels) * tf.math.log(1 - y_pred)
        loss = positive + negative
        if self.gamma > 0:
            pt = labels * y_pred + (1 - labels) * (1 - y_pred)
            loss *= tf.pow(1 - pt, self.gamma)
        return tf.reduce_sum(loss * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)

class FixedWindowAP(tf.keras.callbacks.Callback):
    def __init__(self, sequence: WindowSequence, output_dim: int):
        super().__init__()
        self.sequence = sequence
        self.output_dim = output_dim

    def on_epoch_end(self, epoch, logs=None):
        scored_dim = min(3, self.output_dim)
        labels = [[] for _ in range(scored_dim)]
        scores = [[] for _ in range(scored_dim)]
        for batch in range(len(self.sequence)):
            x, packed = self.sequence[batch]
            pred = np.asarray(self.model.predict_on_batch(x), np.float32)
            y = packed[..., :self.output_dim]
            mask = packed[..., self.output_dim:2 * self.output_dim] > 0
            for target in range(scored_dim):
                labels[target].append(y[..., target][mask[..., target]])
                scores[target].append(pred[..., target][mask[..., target]])
        aps = []
        for target in range(scored_dim):
            yt = np.concatenate(labels[target]) if labels[target] else np.array([])
            pt = np.concatenate(scores[target]) if scores[target] else np.array([])
            if len(yt) and np.unique(yt).size > 1:
                aps.append(float(average_precision_score(yt, pt)))
        # Missing a rare target is penalized rather than silently omitted.
        value = float(np.sum(aps) / scored_dim) if aps else 0.0
        logs = logs if logs is not None else {}
        logs["val_window_map"] = value
        print(f" — val_window_mAP={value:.6f}")

class SamplerState(tf.keras.callbacks.Callback):
    def __init__(self, sequence: WindowSequence, path: Path):
        super().__init__()
        self.sequence = sequence
        self.path = path

    def on_epoch_end(self, epoch, logs=None):
        atomic_json(self.path, {"next_sampler_epoch": int(self.sequence.epoch)})

class AddPosition(tf.keras.layers.Layer):
    def __init__(self, tokens: int, width: int, **kwargs):
        super().__init__(**kwargs)
        self.tokens = tokens
        self.width = width
        self.embedding = tf.keras.layers.Embedding(tokens, width)

    def call(self, x):
        return x + self.embedding(tf.range(tf.shape(x)[1]))

def transformer_block(x, width: int, heads: int, dropout: float, name: str):
    h = tf.keras.layers.LayerNormalization(name=f"{name}_ln1")(x)
    h = tf.keras.layers.MultiHeadAttention(
        num_heads=heads, key_dim=width // heads, dropout=dropout,
        name=f"{name}_mha"
    )(h, h)
    x = tf.keras.layers.Add(name=f"{name}_attn_add")([x, h])
    h = tf.keras.layers.LayerNormalization(name=f"{name}_ln2")(x)
    h = tf.keras.layers.Dense(width * 4, activation=tf.nn.gelu, name=f"{name}_ff1")(h)
    h = tf.keras.layers.Dropout(dropout)(h)
    h = tf.keras.layers.Dense(width, name=f"{name}_ff2")(h)
    return tf.keras.layers.Add(name=f"{name}_ff_add")([x, h])

def build_patch_bilstm(spec: BranchSpec, positive_weights: np.ndarray):
    inputs = tf.keras.Input((CFG.patch_tokens, spec.feature_dim), name="patches")
    x = tf.keras.layers.Dense(CFG.patch_width, name="patch_projection")(inputs)
    x = tf.keras.layers.LayerNormalization()(x)
    x = AddPosition(CFG.patch_tokens, CFG.patch_width, name="position")(x)
    for layer in range(CFG.patch_layers):
        x = transformer_block(x, CFG.patch_width, CFG.patch_heads, .10, f"transformer_{layer}")
    lstm_layers = 1 if RUN_PROFILE in {"smoke", "competition"} else 2
    for layer in range(lstm_layers):
        x = tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(CFG.recurrent_units, return_sequences=True,
                                 dropout=.10), name=f"bilstm_{layer}"
        )(x)
    outputs = tf.keras.layers.Dense(spec.output_dim, activation="sigmoid",
                                    dtype="float32", name="probability")(x)
    model = tf.keras.Model(inputs, outputs, name=f"{spec.domain}_patch_transformer_bilstm")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(3e-4, weight_decay=1e-4, clipnorm=1.0),
        loss=MaskedWeightedBCE(spec.output_dim, positive_weights),
    )
    return model

def build_engineered_bigru(spec: BranchSpec, positive_weights: np.ndarray):
    inputs = tf.keras.Input((None, spec.feature_dim), name="engineered_features")
    x = tf.keras.layers.Dense(64)(inputs)
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(128 if RUN_PROFILE != "smoke" else 48,
                            return_sequences=True, dropout=.10), name="bigru_1")(x)
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(128 if RUN_PROFILE != "smoke" else 48,
                            return_sequences=True, dropout=.10), name="bigru_2")(x)
    x = tf.keras.layers.Dense(256 if RUN_PROFILE != "smoke" else 64, activation="relu")(x)
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.Dropout(.20)(x)
    x = tf.keras.layers.Dense(128 if RUN_PROFILE != "smoke" else 48, activation="relu")(x)
    outputs = tf.keras.layers.Dense(spec.output_dim, activation="sigmoid",
                                    dtype="float32", name="probability")(x)
    model = tf.keras.Model(inputs, outputs, name=f"{spec.domain}_{spec.name}")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(5e-4, weight_decay=1e-4, clipnorm=1.0),
        loss=MaskedWeightedBCE(spec.output_dim, positive_weights),
    )
    return model

def build_wavenet_bigru(spec: BranchSpec, positive_weights: np.ndarray):
    inputs = tf.keras.Input((None, spec.feature_dim), name="engineered_features")
    x = tf.keras.layers.Dense(64)(inputs)
    for block, dilation in enumerate((1, 2, 4)):
        tanh = tf.keras.layers.Conv1D(64, 3, padding="same", dilation_rate=dilation,
                                      activation="tanh", name=f"tanh_{block}")(x)
        gate = tf.keras.layers.Conv1D(64, 3, padding="same", dilation_rate=dilation,
                                      activation="sigmoid", name=f"gate_{block}")(x)
        h = tf.keras.layers.Multiply()([tanh, gate])
        h = tf.keras.layers.Conv1D(64, 1)(h)
        x = tf.keras.layers.Add()([x, h])
    units = 48 if RUN_PROFILE == "smoke" else 128
    for layer in range(2 if RUN_PROFILE == "smoke" else 4):
        x = tf.keras.layers.Bidirectional(
            tf.keras.layers.GRU(units, return_sequences=True), name=f"wave_bigru_{layer}"
        )(x)
        x = tf.keras.layers.Dense(64)(x)
    outputs = tf.keras.layers.Dense(spec.output_dim, activation="sigmoid",
                                    dtype="float32", name="probability")(x)
    model = tf.keras.Model(inputs, outputs, name=f"{spec.domain}_wavenet_bigru")
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(4e-4, weight_decay=1e-4, clipnorm=1.0),
        loss=MaskedWeightedBCE(spec.output_dim, positive_weights, gamma=1.5),
    )
    return model

def build_model(spec: BranchSpec, positive_weights: np.ndarray):
    tf.keras.backend.clear_session()
    if spec.name == "patch_bilstm":
        return build_patch_bilstm(spec, positive_weights)
    if spec.name == "wavenet_bigru":
        return build_wavenet_bigru(spec, positive_weights)
    return build_engineered_bigru(spec, positive_weights)

## 6. Resumable training and streaming overlap-add inference

In [ ]:
import marshal

def atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    os.replace(temp, path)

def code_hash(spec: BranchSpec) -> str:
    digest = hashlib.sha256()
    digest.update(json.dumps(asdict(CFG), sort_keys=True, default=str).encode())
    digest.update(json.dumps(asdict(spec), sort_keys=True).encode())
    digest.update(hashlib.sha256(FOLD_FRAME.to_csv(index=False).encode()).digest())
    for fn in (RecordCache.get, robust_scale, standard_scale, FeatureCache.get,
               extract_padded, choose_center, augment_raw_window, estimate_positive_weights,
               fixed_validation_catalog,
               WindowSequence._new_plan, WindowSequence.__getitem__,
               MaskedWeightedBCE.call, FixedWindowAP.on_epoch_end,
               SamplerState.on_epoch_end,
               transformer_block, build_patch_bilstm,
               build_engineered_bigru, build_wavenet_bigru,
               inference_starts, taper, event_interval_mask,
               streaming_predict_record):
        digest.update(marshal.dumps(fn.__code__))
    for rec in sorted(TRAIN_RECORDS[spec.domain] + TEST_RECORDS[spec.domain]
                      + AUX_RECORDS[spec.domain], key=lambda r: (r.recording_id, r.label_mode)):
        stat = rec.path.stat()
        digest.update(f"{rec.recording_id}:{rec.label_mode}:{stat.st_size}:{stat.st_mtime_ns}".encode())
    event_stat = (META_ROOT / "events.csv").stat()
    digest.update(f"events:{event_stat.st_size}:{event_stat.st_mtime_ns}".encode())
    return digest.hexdigest()[:20]

def inference_starts(n: int, length: int, overlap: float) -> list[int]:
    if n <= length:
        return [0]
    stride = max(1, int(round(length * (1 - overlap))))
    starts = list(range(0, n - length + 1, stride))
    if starts[-1] != n - length:
        starts.append(n - length)
    return starts

def taper(length: int, first: bool, last: bool, center_only: bool) -> np.ndarray:
    if center_only:
        q = max(1, length // 4)
        weight = np.zeros(length, np.float32)
        weight[q:length - q] = 1.0
        if first:
            weight[:q] = 1.0
        if last:
            weight[length - q:] = 1.0
        return weight
    ramp = max(1, int(length * CFG.inference_overlap / 2))
    weight = np.ones(length, np.float32)
    edge = np.linspace(.02, 1.0, ramp, dtype=np.float32)
    weight[:ramp] = edge
    weight[-ramp:] = edge[::-1]
    if first:
        weight[:ramp] = 1.0
    if last:
        weight[-ramp:] = 1.0
    return weight

def event_interval_mask(rec: Record, n: int) -> np.ndarray:
    mask = np.zeros(n, bool)
    rows = EVENTS[EVENTS["Id"].astype(str) == rec.recording_id]
    for _, event in rows.iterrows():
        start, stop = interval_indices(event["Init"], event["Completion"],
                                       DOMAIN[rec.domain]["hz"], n)
        mask[start:stop] = True
    return mask

# Defined before code_hash is first called.
def streaming_predict_record(model: tf.keras.Model, rec: Record,
                             spec: BranchSpec, batch_size: int) -> np.ndarray:
    feat = FEATURE_CACHE.get(rec, spec.name)
    n = len(feat)
    length = min(spec.infer_length, max(spec.train_length, spec.infer_length))
    starts = inference_starts(n, length,
        .50 if spec.name in {"engineered_bigru", "tdcs_turn_specialist", "wavenet_bigru"}
        else CFG.inference_overlap)
    accumulator = np.zeros((n, spec.output_dim), np.float32)
    denominator = np.zeros(n, np.float32)
    for batch_lo in range(0, len(starts), batch_size):
        batch_starts = starts[batch_lo:batch_lo + batch_size]
        windows = []
        for start in batch_starts:
            x = extract_padded(feat, start, length, edge=True)
            if spec.patch_size > 1:
                # Patch model has a fixed 864-token context.
                x = x.reshape(length // spec.patch_size,
                              spec.patch_size * x.shape[1])
            windows.append(x.astype(np.float32))
        pred = np.asarray(model.predict_on_batch(np.stack(windows)), np.float32)
        if spec.patch_size > 1:
            pred = np.repeat(pred, spec.patch_size, axis=1)
        for local, start in enumerate(batch_starts):
            valid = min(length, n - start)
            is_first = start == 0
            is_last = start + length >= n
            weight = taper(length, is_first, is_last,
                           center_only=spec.name != "patch_bilstm")[:valid]
            accumulator[start:start + valid] += pred[local, :valid] * weight[:, None]
            denominator[start:start + valid] += weight
    pred = accumulator / np.maximum(denominator[:, None], 1e-6)
    if spec.output_dim == 4:
        pred = pred[:, :3]
    elif spec.output_dim == 1:
        turn = pred[:, 0].copy()
        pred = np.full((n, 3), EPS, np.float32)
        pred[:, 1] = turn
    if rec.domain == "defog":
        known = event_interval_mask(rec, n)
        pred[~known] = EPS
    return np.clip(pred, EPS, 1 - EPS).astype(np.float32)

def prediction_path(root: Path, rec: Record) -> Path:
    return root / f"{rec.recording_id}.npz"

def save_prediction(path: Path, pred: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(temp, prediction=pred.astype(np.float32))
    os.replace(temp, path)

def load_prediction(path: Path) -> np.ndarray:
    with np.load(path) as data:
        return data["prediction"].astype(np.float32)

class DiskPredictionStore:
    def __init__(self, root: Path):
        self.root = root

    def __getitem__(self, recording_id: str) -> np.ndarray:
        return load_prediction(self.root / f"{recording_id}.npz")

def train_or_load_fold(domain: str, branch: str, fold: int, seed: int):
    spec = branch_spec(domain, branch)
    branch_offset = int(hashlib.sha256(f"{domain}:{branch}".encode()).hexdigest()[:8], 16)
    fit_seed = int((seed + 1009 * fold + branch_offset) % (2**31 - 1))
    seed_everything(fit_seed)
    train_records, train_catalog = get_catalog(domain, fold, "train")
    val_records, val_catalog = get_catalog(domain, fold, "validation")
    if not train_records or not val_records:
        raise RuntimeError(f"Empty fold split: {domain}/{fold}")
    positive_weights = estimate_positive_weights(
        train_records, spec.output_dim, spec.full_inverse
    )
    run_dir = ARTIFACT_ROOT / "neural" / domain / branch / f"fold_{fold}" / f"seed_{seed}"
    weights_path = run_dir / "best.weights.h5"
    done_path = run_dir / "done.json"
    fingerprint = code_hash(spec)
    progress_path = run_dir / "in_progress.json"
    model = build_model(spec, positive_weights)
    reusable = False
    if REUSE_COMPLETED and done_path.exists() and weights_path.exists():
        saved = json.loads(done_path.read_text(encoding="utf-8"))
        reusable = saved.get("fingerprint") == fingerprint
    if reusable:
        model.load_weights(weights_path)
        print(f"reuse {domain}/{branch}/fold{fold}/seed{seed}")
    else:
        if (run_dir / "fit_backup").exists():
            if not progress_path.exists():
                raise RuntimeError(f"Unfingerprinted fit backup at {run_dir}; remove or inspect it.")
            progress = json.loads(progress_path.read_text(encoding="utf-8"))
            if progress.get("fingerprint") != fingerprint:
                raise RuntimeError(
                    f"Stale fit backup at {run_dir}. Move it aside before running changed code."
                )
        atomic_json(progress_path, {"fingerprint": fingerprint, "started_utc": pd.Timestamp.utcnow().isoformat()})
        batch = CFG.batch_patch if branch == "patch_bilstm" else CFG.batch_bigru
        steps = CFG.steps_per_epoch
        if RUN_PROFILE == "smoke":
            steps = min(steps, 24)
        train_seq = WindowSequence(
            train_records, train_catalog, spec, steps, batch,
            seed + fold * 1009, training=True, augment=CFG.augment
        )
        sampler_state_path = run_dir / "sampler_state.json"
        if sampler_state_path.exists() and (run_dir / "fit_backup").exists():
            saved_sampler_epoch = int(json.loads(
                sampler_state_path.read_text(encoding="utf-8")
            ).get("next_sampler_epoch", train_seq.epoch))
            train_seq.epoch = saved_sampler_epoch
            train_seq.plan = train_seq._new_plan()
            train_seq.epoch += 1
        val_spec = replace(spec, positive_fraction=0.0)
        validation_catalog = fixed_validation_catalog(val_records, val_catalog)
        val_seq = WindowSequence(
            val_records, validation_catalog, val_spec, max(12, min(64, steps // 2)), batch,
            seed + fold * 1009 + 777, training=False
        )
        run_dir.mkdir(parents=True, exist_ok=True)
        epoch_best = run_dir / "epoch_best.weights.h5"
        started = time.time()
        history = model.fit(
            train_seq, validation_data=val_seq,
            epochs=CFG.epochs_patch if branch == "patch_bilstm" else CFG.epochs_bigru,
            callbacks=[
                FixedWindowAP(val_seq, spec.output_dim),
                tf.keras.callbacks.ModelCheckpoint(
                    epoch_best, monitor="val_window_map", mode="max",
                    save_best_only=True, save_weights_only=True,
                ),
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_window_map", mode="max",
                    patience=1 if RUN_PROFILE == "smoke" else 5,
                    restore_best_weights=True,
                ),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss", mode="min", factor=.5,
                    patience=1 if RUN_PROFILE == "smoke" else 2,
                    min_lr=2e-6,
                ),
                tf.keras.callbacks.BackupAndRestore(
                    backup_dir=run_dir / "fit_backup", delete_checkpoint=True,
                ),
                SamplerState(train_seq, sampler_state_path),
                tf.keras.callbacks.CSVLogger(run_dir / "history.csv", append=True),
            ],
            verbose=2,
        )
        if epoch_best.exists():
            model.load_weights(epoch_best)
        temp_weights = run_dir / "best.tmp.weights.h5"
        model.save_weights(temp_weights)
        os.replace(temp_weights, weights_path)
        atomic_json(done_path, {
            "fingerprint": fingerprint, "domain": domain, "branch": branch,
            "fold": fold, "seed": seed, "positive_weights": positive_weights.tolist(),
            "epochs_ran": len(history.history.get("loss", [])),
            "elapsed_seconds": time.time() - started,
        })
        if progress_path.exists():
            progress_path.unlink()
    return model, spec, val_records, reusable

In [ ]:
def ensure_fold_predictions(model, spec: BranchSpec, records: list[Record], root: Path,
                            batch_size: int, force: bool = False) -> None:
    for number, rec in enumerate(records, 1):
        path = prediction_path(root, rec)
        if path.exists() and not force:
            try:
                existing = load_prediction(path)
                if existing.shape == (rec.n_rows, 3) and np.isfinite(existing).all():
                    continue
            except Exception:
                pass
        pred = streaming_predict_record(model, rec, spec, batch_size)
        save_prediction(path, pred)
        if number % 20 == 0:
            print(f"  predicted {number}/{len(records)} records")

def average_prediction_files(paths: list[Path]) -> np.ndarray:
    if not paths:
        raise RuntimeError("No fold predictions to average")
    total = None
    for path in paths:
        pred = load_prediction(path)
        total = pred.astype(np.float64) if total is None else total + pred
    return (total / len(paths)).astype(np.float32)

PREDICTION_INDEX: dict[str, dict[str, dict[str, DiskPredictionStore]]] = {
    d: {} for d in DOMAIN
}

# Persist the event specialist in the same per-record interface as neural branches.
event_root = ARTIFACT_ROOT / "predictions" / "defog" / "event_specialist"
for split, records, store in (
    ("oof", TRAIN_RECORDS["defog"], EVENT_BASE_OOF_STORE),
    ("test", TEST_RECORDS["defog"], EVENT_BASE_TEST_STORE),
):
    for rec in records:
        save_prediction(event_root / split / f"{rec.recording_id}.npz", store[rec.recording_id])
PREDICTION_INDEX["defog"]["event_specialist"] = {
    "oof": DiskPredictionStore(event_root / "oof"),
    "test": DiskPredictionStore(event_root / "test"),
}
del EVENT_BASE_OOF_STORE, EVENT_BASE_TEST_STORE, TASK_PRIOR_OOF_STORE
gc.collect()

if RUN_NEURAL:
    for domain in DOMAIN:
        branches = [b for b in CFG.active_branches
                    if not (b == "tdcs_turn_specialist" and domain != "tdcsfog")]
        for branch in branches:
            print(f"\n### {domain} / {branch}")
            for fold in CFG.folds:
                for seed in CFG.seeds:
                    if (time.time() - RUN_STARTED) / 3600 > MAX_WALL_HOURS:
                        raise TimeoutError(
                            "Runtime guard reached. Re-run the notebook: completed folds will resume "
                            "from fingerprinted artifacts."
                        )
                    model, spec, val_records, reused = train_or_load_fold(domain, branch, fold, seed)
                    run_dir = ARTIFACT_ROOT / "neural" / domain / branch / f"fold_{fold}" / f"seed_{seed}"
                    infer_batch = (1 if branch == "patch_bilstm" or domain == "defog"
                                   else max(1, CFG.batch_bigru // 6))
                    ensure_fold_predictions(model, spec, val_records,
                                            run_dir / "oof", infer_batch, force=not reused)
                    ensure_fold_predictions(model, spec, TEST_RECORDS[domain],
                                            run_dir / "test", infer_batch, force=not reused)
                    del model
                    tf.keras.backend.clear_session()
                    FEATURE_CACHE.clear(); CACHE.clear(); gc.collect()

            branch_root = ARTIFACT_ROOT / "predictions" / domain / branch
            # OOF: each recording comes from its held-out fold, averaged over seeds.
            for rec in TRAIN_RECORDS[domain]:
                fold = FOLD_BY_RECORD[domain][rec.recording_id]
                paths = [ARTIFACT_ROOT / "neural" / domain / branch / f"fold_{fold}" /
                         f"seed_{seed}" / "oof" / f"{rec.recording_id}.npz"
                         for seed in CFG.seeds]
                if not all(path.exists() for path in paths):
                    raise RuntimeError(f"Incomplete OOF artifacts for {domain}/{branch}/{rec.recording_id}")
                save_prediction(branch_root / "oof" / f"{rec.recording_id}.npz",
                                average_prediction_files(paths))
            # Test: average all independent fold/seed models per recording.
            for rec in TEST_RECORDS[domain]:
                paths = [ARTIFACT_ROOT / "neural" / domain / branch / f"fold_{fold}" /
                         f"seed_{seed}" / "test" / f"{rec.recording_id}.npz"
                         for fold in CFG.folds for seed in CFG.seeds]
                if not all(path.exists() for path in paths):
                    raise RuntimeError(f"Incomplete test artifacts for {domain}/{branch}/{rec.recording_id}")
                save_prediction(branch_root / "test" / f"{rec.recording_id}.npz",
                                average_prediction_files(paths))
            PREDICTION_INDEX[domain][branch] = {
                "oof": DiskPredictionStore(branch_root / "oof"),
                "test": DiskPredictionStore(branch_root / "test"),
            }

branch_metrics = []
for domain, branches in PREDICTION_INDEX.items():
    for name, stores in branches.items():
        branch_metrics.append({"domain": domain, "branch": name,
                               **exact_ap(TRAIN_RECORDS[domain], stores["oof"])})
BRANCH_METRICS = pd.DataFrame(branch_metrics)
BRANCH_METRICS.to_csv(ARTIFACT_ROOT / "branch_oof_metrics.csv", index=False)
display(BRANCH_METRICS)

Every native prediction is written immediately to a compressed file.
The overlap accumulator exists for one recording only; the notebook
never stores all overlapping window outputs or all daily recordings in
RAM.  A completed fold is reused only when its configuration and actual
function bytecode fingerprint match.

## 7. Pool OOF neural evidence into the DeFOG event classifier

In [ ]:
def add_event_neural_features(events: pd.DataFrame, split: str) -> pd.DataFrame:
    out = events.copy()
    neural_names = [name for name in PREDICTION_INDEX["defog"]
                    if name != "event_specialist"]
    for branch in neural_names:
        store = PREDICTION_INDEX["defog"][branch][split]
        pooled_mean = np.zeros((len(out), 3), np.float32)
        pooled_max = np.zeros((len(out), 3), np.float32)
        for pos, (_, event) in enumerate(out.iterrows()):
            pred = store[event["recording_id"]]
            segment = pred[int(event["start"]):int(event["stop"])]
            if len(segment):
                pooled_mean[pos] = segment.mean(axis=0)
                pooled_max[pos] = segment.max(axis=0)
        for target, name in enumerate(TARGETS):
            out[f"neural_{branch}_{name}_mean"] = pooled_mean[:, target]
            out[f"neural_{branch}_{name}_max"] = pooled_max[:, target]
    return out

if (RUN_NEURAL and RUN_EXPLORATORY_META_BLEND
        and any(name != "event_specialist" for name in PREDICTION_INDEX["defog"])):
    event_train_neural = add_event_neural_features(DEFOG_EVENT_TRAIN_FIT, "oof")
    event_test_neural = add_event_neural_features(DEFOG_EVENT_TEST, "test")
    EVENT_NEURAL_OOF, EVENT_NEURAL_TEST, EVENT_NEURAL_MODEL, EVENT_NEURAL_FOLDS = (
        crossfit_event_specialist(event_train_neural, event_test_neural)
    )
    event_neural_oof_store = broadcast_events(
        TRAIN_RECORDS["defog"], event_train_neural, EVENT_NEURAL_OOF
    )
    event_neural_test_store = broadcast_events(
        TEST_RECORDS["defog"], event_test_neural, EVENT_NEURAL_TEST
    )
    root = ARTIFACT_ROOT / "predictions" / "defog" / "event_plus_neural"
    for split, records, store in (
        ("oof", TRAIN_RECORDS["defog"], event_neural_oof_store),
        ("test", TEST_RECORDS["defog"], event_neural_test_store),
    ):
        for rec in records:
            save_prediction(root / split / f"{rec.recording_id}.npz", store[rec.recording_id])
    PREDICTION_INDEX["defog"]["event_plus_neural"] = {
        "oof": DiskPredictionStore(root / "oof"),
        "test": DiskPredictionStore(root / "test"),
    }
    print("Event + cross-fitted neural OOF:",
          exact_ap(TRAIN_RECORDS["defog"], PREDICTION_INDEX["defog"]["event_plus_neural"]["oof"]))
    del event_neural_oof_store, event_neural_test_store
    gc.collect()
else:
    print("Nested neural event stacking is off; clean event-specialist OOF remains the primary DeFOG estimate.")

This exploratory stack is disabled by default.  Ordinary OOF base
predictions are not sufficient for a strictly nested meta-model: inner
base models used for meta-training can still have seen the outer held-out
subjects.  Turn it on only as a diagnostic, never present its score as
clean OOF.  A publication-grade stack needs inner subject folds inside
every outer fold.

## 8. Predeclared evidence-weighted branch blending

In [ ]:
# These weights are declared before observing this notebook's OOF scores.
# They encode external top-solution evidence plus the local TDCS Turn
# specialist result, avoiding a second-level OOF selection leak.
PREDECLARED_WEIGHTS = {
    "defog": {
        "StartHesitation": {"event_specialist": .80, "patch_bilstm": .12, "engineered_bigru": .08},
        "Turn": {"event_specialist": .85, "patch_bilstm": .10, "engineered_bigru": .05},
        "Walking": {"event_specialist": .75, "patch_bilstm": .15, "engineered_bigru": .10},
    },
    "tdcsfog": {
        "StartHesitation": {"patch_bilstm": .50, "engineered_bigru": .50},
        "Turn": {"patch_bilstm": .375, "engineered_bigru": .375, "tdcs_turn_specialist": .25},
        "Walking": {"patch_bilstm": .50, "engineered_bigru": .50},
    },
}

def fixed_weights(domain: str, target_name: str):
    available = PREDICTION_INDEX[domain]
    declared = PREDECLARED_WEIGHTS[domain][target_name]
    kept = {name: weight for name, weight in declared.items() if name in available}
    # Event-only mode stays useful without forcing neural completion.
    if not kept and len(available) == 1:
        kept = {next(iter(available)): 1.0}
    if not kept:
        raise RuntimeError(f"No declared branch available for {domain}/{target_name}")
    total = sum(kept.values())
    return list(kept), np.array([kept[name] / total for name in kept], np.float32)

BLEND_WEIGHTS = []
BLENDED_STORES = {}
for domain in DOMAIN:
    if not PREDICTION_INDEX[domain]:
        continue
    blend_root = ARTIFACT_ROOT / "predictions" / domain / "evidence_weighted_blend"
    # Start per-record arrays, then fill one target at a time.
    oof_arrays = {r.recording_id: np.zeros((r.n_rows, 3), np.float32)
                  for r in TRAIN_RECORDS[domain]}
    test_arrays = {r.recording_id: np.zeros((r.n_rows, 3), np.float32)
                   for r in TEST_RECORDS[domain]}
    for target, target_name in enumerate(TARGETS):
        names, weights = fixed_weights(domain, target_name)
        BLEND_WEIGHTS.extend({
            "domain": domain, "target": target_name, "source": "predeclared_external_evidence",
            "branch": name, "weight": float(weights[j]),
        } for j, name in enumerate(names))
        for rec in TRAIN_RECORDS[domain]:
            matrix = np.column_stack([
                PREDICTION_INDEX[domain][name]["oof"][rec.recording_id][:, target]
                for name in names
            ])
            oof_arrays[rec.recording_id][:, target] = matrix @ weights
        for rec in TEST_RECORDS[domain]:
            matrix = np.column_stack([
                PREDICTION_INDEX[domain][name]["test"][rec.recording_id][:, target]
                for name in names
            ])
            test_arrays[rec.recording_id][:, target] = matrix @ weights

    for rec in TRAIN_RECORDS[domain]:
        save_prediction(blend_root / "oof" / f"{rec.recording_id}.npz",
                        np.clip(oof_arrays[rec.recording_id], EPS, 1 - EPS))
    for rec in TEST_RECORDS[domain]:
        save_prediction(blend_root / "test" / f"{rec.recording_id}.npz",
                        np.clip(test_arrays[rec.recording_id], EPS, 1 - EPS))
    BLENDED_STORES[domain] = {
        "oof": DiskPredictionStore(blend_root / "oof"),
        "test": DiskPredictionStore(blend_root / "test"),
    }
    del oof_arrays, test_arrays
    gc.collect()

BLEND_WEIGHT_FRAME = pd.DataFrame(BLEND_WEIGHTS)
BLEND_WEIGHT_FRAME.to_csv(ARTIFACT_ROOT / "predeclared_blend_weights.csv", index=False)
display(BLEND_WEIGHT_FRAME.pivot_table(
    index=["domain", "target"], columns="branch", values="weight", fill_value=0
))

def fold_metric_table(domain: str, store: DiskPredictionStore) -> pd.DataFrame:
    rows = []
    for fold in range(CFG.n_folds):
        records = [r for r in TRAIN_RECORDS[domain]
                   if FOLD_BY_RECORD[domain][r.recording_id] == fold]
        if records:
            rows.append({"domain": domain, "fold": fold, **exact_ap(records, store)})
    return pd.DataFrame(rows)

blend_metrics = []
fold_metrics = []
for domain, stores in BLENDED_STORES.items():
    blend_metrics.append({"domain": domain, **exact_ap(TRAIN_RECORDS[domain], stores["oof"])})
    fold_metrics.append(fold_metric_table(domain, stores["oof"]))
BLEND_METRICS = pd.DataFrame(blend_metrics)
FOLD_METRICS = pd.concat(fold_metrics, ignore_index=True) if fold_metrics else pd.DataFrame()
BLEND_METRICS.to_csv(ARTIFACT_ROOT / "evidence_weighted_oof_metrics.csv", index=False)
FOLD_METRICS.to_csv(ARTIFACT_ROOT / "evidence_weighted_fold_metrics.csv", index=False)
display(BLEND_METRICS)
display(FOLD_METRICS)

In [ ]:
def pooled_exact_ap(stores_by_domain: dict[str, DiskPredictionStore]) -> dict[str, float]:
    ys, ps = [], []
    for domain, store in stores_by_domain.items():
        for rec in TRAIN_RECORDS[domain]:
            item = CACHE.get(rec)
            mask = item["score_mask"]
            ys.append(item["labels"][mask])
            ps.append(store[rec.recording_id][mask])
    y, p = np.concatenate(ys), np.concatenate(ps)
    result = {name: float(average_precision_score(y[:, i], p[:, i]))
              for i, name in enumerate(TARGETS)}
    result["mAP"] = float(np.mean(list(result.values())))
    return result

def scale_sensitivity_audit(stores_by_domain: dict[str, DiskPredictionStore]) -> pd.DataFrame:
    if set(stores_by_domain) != set(DOMAIN):
        return pd.DataFrame()
    rows = []
    for factor in (.50, .75, 1.0, 1.25, 1.50, 2.0):
        class ScaledStore:
            def __init__(self, base, scale): self.base, self.scale = base, scale
            def __getitem__(self, rid): return np.clip(self.base[rid] * self.scale, EPS, 1 - EPS)
        scaled = {
            "defog": stores_by_domain["defog"],
            "tdcsfog": ScaledStore(stores_by_domain["tdcsfog"], factor),
        }
        rows.append({"tdcs_scale_diagnostic_only": factor, **pooled_exact_ap(scaled)})
    return pd.DataFrame(rows)

if set(BLENDED_STORES) == set(DOMAIN):
    POOLED_OOF = pooled_exact_ap({d: BLENDED_STORES[d]["oof"] for d in DOMAIN})
    print("Pooled native-row OOF for the predeclared blend:")
    display(pd.DataFrame([POOLED_OOF]))
    SCALE_AUDIT = scale_sensitivity_audit({d: BLENDED_STORES[d]["oof"] for d in DOMAIN})
    display(SCALE_AUDIT)
    SCALE_AUDIT.to_csv(ARTIFACT_ROOT / "cross_domain_scale_sensitivity_DO_NOT_TUNE.csv", index=False)
else:
    POOLED_OOF = {}
    SCALE_AUDIT = pd.DataFrame()
    print("Pooled OOF requires completed sensor branches for both domains.")

# These are diagnostics only; blend weights were declared above rather
# than selected from these OOF values or a public leaderboard score.
if not FOLD_METRICS.empty:
    display(FOLD_METRICS.groupby("domain")[[*TARGETS, "mAP"]].agg(["mean", "std", "min", "max"]))

## 9. Strict submission construction

In [ ]:
def locate_sample_submission() -> Path:
    candidates = [
        DATA_ROOT / "sample_submission.csv",
        DATA_ROOT / "submissions" / "sample_submission.csv",
        PROJECT_ROOT / "data" / "submissions" / "sample_submission.csv",
    ]
    return next((p for p in candidates if p.exists()), None)

# Fixed cross-domain calibration selected from the completed OOF
# sensitivity audit. This changes only TDCS-vs-DeFOG ranking; it does
# not change ranking within either domain.
SUBMISSION_DOMAIN_SCALE = {
    "defog": 1.00,
    "tdcsfog": 0.50,
}

SAMPLE_PATH = locate_sample_submission()
SUBMISSION_PATH = None
if SAMPLE_PATH is None:
    print("No sample_submission.csv found; skipping ID contract validation.")
elif set(BLENDED_STORES) != set(DOMAIN):
    print("Submission skipped: finish all domain branches first.")
else:
    parts = []
    for domain in DOMAIN:
        for rec in TEST_RECORDS[domain]:
            item = CACHE.get(rec)
            pred = BLENDED_STORES[domain]["test"][rec.recording_id].copy()
            pred = np.clip(
                pred * SUBMISSION_DOMAIN_SCALE[domain],
                EPS,
                1 - EPS,
            )
            if pred.shape != (len(item["time"]), 3):
                raise ValueError(f"Bad test shape for {rec.recording_id}: {pred.shape}")
            part = pd.DataFrame(pred, columns=TARGETS)
            part.insert(0, "Id", [f"{rec.recording_id}_{int(t)}" for t in item["time"]])
            parts.append(part)
    candidate = pd.concat(parts, ignore_index=True)
    sample = pd.read_csv(SAMPLE_PATH)
    required = ["Id", *TARGETS]
    if list(sample.columns) != required:
        raise ValueError(f"Unexpected sample columns: {sample.columns.tolist()}")
    if candidate["Id"].duplicated().any():
        raise ValueError("Generated duplicate submission IDs")
    sample_ids, candidate_ids = set(sample["Id"]), set(candidate["Id"])
    if sample_ids != candidate_ids:
        raise ValueError(
            f"Submission ID mismatch: missing={len(sample_ids-candidate_ids)}, "
            f"extra={len(candidate_ids-sample_ids)}"
        )
    submission = candidate.set_index("Id").reindex(sample["Id"]).reset_index()
    values = submission[TARGETS].to_numpy(float)
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Submission probabilities must be finite and in [0,1]")
    filename = "submission.csv" if RUN_FINAL_SUBMISSION else "submission_smoke_DO_NOT_SUBMIT.csv"
    if Path("/kaggle/working").exists():
        SUBMISSION_PATH = Path("/kaggle/working") / filename
    else:
        SUBMISSION_PATH = ARTIFACT_ROOT / filename
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Wrote {len(submission):,} rows to {SUBMISSION_PATH}")
    display(submission.head())
    display(submission[TARGETS].describe(percentiles=[.001, .01, .5, .99, .999]))
    if not RUN_FINAL_SUBMISSION:
        print("SMOKE OUTPUT ONLY — set FOG_PROFILE=competition and FOG_RUN_SUBMISSION=1 for a real entry.")

## 10. Reproducibility manifest and promotion checklist

In [ ]:
data_inventory = []
for domain in DOMAIN:
    for split, records in (("train", TRAIN_RECORDS[domain]), ("test", TEST_RECORDS[domain]),
                           ("event_only_aux", AUX_RECORDS[domain])):
        for rec in records:
            data_inventory.append({
                "domain": domain, "split": split, "recording_id": rec.recording_id,
                "subject": rec.subject, "rows": rec.n_rows,
                "bytes": rec.path.stat().st_size,
            })
inventory_frame = pd.DataFrame(data_inventory)
inventory_hash = hashlib.sha256(
    inventory_frame.sort_values(["domain", "split", "recording_id"])
    .to_csv(index=False).encode()
).hexdigest()
manifest = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "profile": asdict(CFG),
    "run_neural": RUN_NEURAL,
    "run_exploratory_meta_blend": RUN_EXPLORATORY_META_BLEND,
    "run_final_submission": RUN_FINAL_SUBMISSION,
    "submission_domain_scale": SUBMISSION_DOMAIN_SCALE,
    "versions": {"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
                 "scipy": scipy.__version__, "sklearn": sklearn.__version__,
                 "tensorflow": tf.__version__},
    "data_inventory_sha256": inventory_hash,
    "fold_manifest_sha256": hashlib.sha256(FOLD_FRAME.to_csv(index=False).encode()).hexdigest(),
    "available_branches": {d: list(v) for d, v in PREDICTION_INDEX.items()},
    "pooled_oof": POOLED_OOF,
    "submission": str(SUBMISSION_PATH) if SUBMISSION_PATH else None,
}
atomic_json(ARTIFACT_ROOT / "run_manifest.json", manifest)
display(pd.DataFrame([
    {"candidate": "focal loss", "default": False, "reason": "top-team diversity; inconsistent local OOF"},
    {"candidate": "scale/rotation, drift, and channel ablation", "default": CFG.augment,
     "reason": "third-place recipe; promote only on repeated-fold gain"},
    {"candidate": "hard type pseudo-labels", "default": False,
     "reason": "second-place method; error amplification risk"},
    {"candidate": "event-only auxiliary supervision", "default": bool(AUX_RECORDS["defog"]),
     "reason": "first-place masked-head method; safe when Event exists"},
    {"candidate": "WaveNet–BiGRU focal branch", "default": False,
     "reason": "implemented fifth-place diversity ablation; enable only after clean OOF promotion"},
    {"candidate": "Daily Living SSL", "default": False,
     "reason": "costly and inconsistent; requires its own OOF ablation"},
    {"candidate": "per-domain Platt scaling", "default": False,
     "reason": "can alter pooled ranks; previous notebook tuned it optimistically"},
]))
print("Manifest:", ARTIFACT_ROOT / "run_manifest.json")

## Reading the result

A credible run has all of the following:

- five immutable subject folds and every fold artifact present;
- native-row AP for each domain, each target, and the pooled competition
  metric—not binary accuracy or a window metric;
- predeclared blend weights that were not fitted to these OOF rows;
- fold dispersion shown beside pooled AP;
- exact equality between generated IDs and `sample_submission.csv`;
- a `submission.csv` created only by a non-smoke profile.

Do not claim the smoke score, one public leaderboard submission, or a
post-hoc blend selected on the same OOF rows as proof of improvement.

### Primary sources behind the recipe

- [Organizer analysis of the competition and top solutions](https://www.nature.com/articles/s41467-024-49027-0)
- [First-place Transformer–BiLSTM write-up](https://www.kaggle.com/competitions/tlvmc-parkinsons-freezing-gait-prediction/writeups/baurzhan-urazalinov-1st-place-solution-transformer)
- [Second-place engineered BiGRU write-up](https://www.kaggle.com/competitions/tlvmc-parkinsons-freezing-gait-prediction/writeups/waiwai-2nd-place-solution)
- [Third-place Transformer/GRU augmentation write-up](https://www.kaggle.com/competitions/tlvmc-parkinsons-freezing-gait-prediction/writeups/stochoshi-3rd-place-solution-transformer-gru)
- [Fifth-place WaveNet–GRU discussion](https://www.kaggle.com/competitions/tlvmc-parkinsons-freezing-gait-prediction/discussion/418275)
- [Archived top-solution code and models](https://zenodo.org/records/12565292)

### Kaggle execution and resumption

`competition` is deliberately not the Run-All default: that prevents an
accidental multi-hour job.  In the first cell, set the visible
`FOG_PROFILE` default to `competition` (or set the environment variable)
and set `FOG_RUN_SUBMISSION=1`.  For runtime control, a job can be sharded
by editing `folds`, `active_branches`, or the domain loop; attach the
resulting `fog-evidence-weighted` artifact directory as a Kaggle Dataset
to resume in a later session.  `/kaggle/working` itself is not persistent.

The main remaining score risk is **cross-domain score alignment**:
competition AP pools DeFOG and TDCS rows.  The final submission applies
the fixed `SUBMISSION_DOMAIN_SCALE` declared in Section 9.  Because its
TDCS value was selected from this run's OOF sensitivity table rather than
nested inner folds, treat the apparent OOF lift as optimistic and retain
the unscaled submission as a conservative alternative.